01. Librerias y funciones

In [3]:
# ==================== IMPORTS Y CONFIGURACIÓN INICIAL ====================
import pandas as pd
import folium
from folium.features import CustomIcon, DivIcon
from folium.plugins import MarkerCluster, HeatMap
from shapely.geometry import MultiPoint, Point
import os
import glob
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import ScatterChart, Reference, Series
import warnings
warnings.filterwarnings('ignore')


02. Generador de Mapa o Excel

In [4]:
# ================== MAPA DESDE MATRIZ PRECALCULADA ==================

# Buscar archivos Excel en la carpeta de inputs
input_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input_vol_import'
archivos_excel = glob.glob(os.path.join(input_dir, '*.xlsx')) + glob.glob(os.path.join(input_dir, '*.xlsm'))

if archivos_excel:
    archivo_matriz = archivos_excel[0]  # Tomar el primer archivo encontrado
else:
    archivo_matriz = None

output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
icon_met_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')

# Leer y explorar el archivo Excel
if archivo_matriz:
    try:
        # Obtener todas las hojas disponibles
        excel_file = pd.ExcelFile(archivo_matriz)
        hojas_disponibles = excel_file.sheet_names
        
        # Buscar hoja que contenga datos de recorridos (buscar palabras clave)
        hoja_seleccionada = None
        palabras_clave = ['recorrido', 'ruta', 'cliente', 'met', 'datos']
        
        for hoja in hojas_disponibles:
            hoja_lower = hoja.lower()
            if any(palabra in hoja_lower for palabra in palabras_clave):
                hoja_seleccionada = hoja
                break
        
        # Si no se encuentra por palabras clave, usar la primera hoja
        if not hoja_seleccionada:
            hoja_seleccionada = hojas_disponibles[0]
        
        
        # Leer la hoja seleccionada
        df_recorridos = pd.read_excel(archivo_matriz, sheet_name=hoja_seleccionada)
            
    except Exception as e:
        print(f"❌ Error al leer el archivo Excel: {e}")
        df_recorridos = None
else:
    print("❌ No se encontró archivo Excel para procesar")
    df_recorridos = None

# Verificar que el DataFrame fue cargado correctamente antes de continuar
if df_recorridos is not None and not df_recorridos.empty:
    # Mapeo automático de columnas
    def encontrar_columna(palabras_clave, columnas_disponibles):
        for col in columnas_disponibles:
            col_clean = col.lower().replace('🏠', '').replace('🛣️', '').replace('🔢', '').replace('🆔', '').replace('🌍', '').replace('📚', '').replace('📏', '').strip()
            for palabra in palabras_clave:
                if palabra.lower() in col_clean:
                    return col
        return None
    
    # Buscar columnas automáticamente
    col_met = encontrar_columna(['met'], df_recorridos.columns) or 'MET'
    col_ruta = encontrar_columna(['ruta'], df_recorridos.columns) or 'Ruta'
    col_secuencia = encontrar_columna(['secuencia'], df_recorridos.columns) or 'Secuencia'
    col_tipo = encontrar_columna(['tipo'], df_recorridos.columns) or 'Tipo'
    col_codigo = encontrar_columna(['codigo'], df_recorridos.columns) or '🆔 Codigo'
    col_longitud = encontrar_columna(['longitud'], df_recorridos.columns) or '🌍 Longitud'
    col_latitud = encontrar_columna(['latitud'], df_recorridos.columns) or '🌍 Latitud'
    col_nombre = encontrar_columna(['nombre'], df_recorridos.columns) or '📚 Nombre'
    col_distancia_sig = encontrar_columna(['distancia al siguiente'], df_recorridos.columns) or '📏 Distancia_al_siguiente_km'
    col_volumen = encontrar_columna(['volumen promedio diario', 'volumen'], df_recorridos.columns) or 'Volumen promedio diario'
    col_importe = encontrar_columna(['importe promedio diario', 'importe'], df_recorridos.columns) or 'Importe promedio diario'
    col_tiros = encontrar_columna(['tiros promedio diario', 'tiros'], df_recorridos.columns) or 'Tiros promedio diario'
    
    # NUEVAS COLUMNAS AGREGADAS
    col_volumen_promedio_ruta = encontrar_columna(['volumen promedio diario por ruta'], df_recorridos.columns) or 'Volumen promedio diario por Ruta'
    col_volumen_min_ruta = encontrar_columna(['volumen min por ruta', 'volumen mín por ruta'], df_recorridos.columns) or 'Volumen MIN por Ruta'
    col_volumen_max_ruta = encontrar_columna(['volumen max por ruta', 'volumen máx por ruta'], df_recorridos.columns) or 'Volumen MAX por Ruta'
    col_importe_promedio_ruta = encontrar_columna(['importe promedio diario por ruta'], df_recorridos.columns) or 'Importe promedio diario por Ruta'
    col_distancia_promedio_ruta = encontrar_columna(['distancia promedio diaria por ruta'], df_recorridos.columns) or 'Distancia promedio diaria por Ruta'
    col_tiros_promedio_ruta = encontrar_columna(['tiros promedio diario por ruta'], df_recorridos.columns) or 'Tiros promedio diario por Ruta'
    
    # NUEVAS COLUMNAS DE MET
    col_volumen_promedio_met = encontrar_columna(['volumen promedio diaria por met'], df_recorridos.columns) or 'Volumen promedio diaria por MET'
    col_importe_promedio_met = encontrar_columna(['importe promedio diaria por met'], df_recorridos.columns) or 'Importe promedio diaria por MET'
    col_distancia_promedio_met = encontrar_columna(['distancia promedio diaria por met'], df_recorridos.columns) or 'Distancia promedio diaria por MET'
    col_tiros_promedio_met = encontrar_columna(['tiros promedio diario por met'], df_recorridos.columns) or 'Tiros promedio diario por MET'
    
    
    # Verificar qué columnas existen realmente
    columnas_requeridas = [col_met, col_ruta, col_secuencia, col_tipo, col_codigo, 
                          col_longitud, col_latitud, col_nombre]
    columnas_opcionales = [col_distancia_sig, col_volumen, col_importe, col_tiros,
                          col_volumen_promedio_ruta, col_volumen_min_ruta, col_volumen_max_ruta,
                          col_importe_promedio_ruta, col_distancia_promedio_ruta, col_tiros_promedio_ruta,
                          col_volumen_promedio_met, col_importe_promedio_met, col_distancia_promedio_met, col_tiros_promedio_met]
    
    # Verificar columnas requeridas
    columnas_faltantes = []
    for col in columnas_requeridas:
        if col not in df_recorridos.columns:
            columnas_faltantes.append(col)
    
    if columnas_faltantes:
        print(f"\n❌ FALTAN COLUMNAS CRÍTICAS: {columnas_faltantes}")
        print("🔍 Columnas disponibles en el archivo:")
        for col in df_recorridos.columns:
            print(f"   - {col}")
        df_recorridos = None  # Marcar como inválido
    else:
        # Todas las columnas críticas están disponibles
        
        # Función para normalizar y limpiar datos numéricos
        def limpiar_numerico(valor):
            if pd.isna(valor) or valor == '' or valor == 0:
                return 0
            try:
                return float(valor)
            except:
                return 0

        # Limpiar y normalizar columnas numéricas si existen
        if col_volumen in df_recorridos.columns:
            df_recorridos[col_volumen] = df_recorridos[col_volumen].apply(limpiar_numerico)
        else:
            # Crear columna con valores por defecto
            df_recorridos[col_volumen] = 1.0
            print(f"⚠️ Columna {col_volumen} creada con valores por defecto")
            
        if col_importe in df_recorridos.columns:
            df_recorridos[col_importe] = df_recorridos[col_importe].apply(limpiar_numerico)
        else:
            # Crear columna con valores por defecto
            df_recorridos[col_importe] = 100.0
            print(f"⚠️ Columna {col_importe} creada con valores por defecto")
        
        if col_tiros in df_recorridos.columns:
            df_recorridos[col_tiros] = df_recorridos[col_tiros].apply(limpiar_numerico)
        else:
            # Crear columna con valores por defecto
            df_recorridos[col_tiros] = 1.0
            print(f"⚠️ Columna {col_tiros} creada con valores por defecto")
            
        if col_distancia_sig not in df_recorridos.columns:
            df_recorridos[col_distancia_sig] = 0.0
            print(f"⚠️ Columna {col_distancia_sig} creada con valores por defecto")

        # LIMPIAR NUEVAS COLUMNAS AGREGADAS
        for nueva_col in [col_volumen_promedio_ruta, col_volumen_min_ruta, col_volumen_max_ruta, 
                         col_importe_promedio_ruta, col_distancia_promedio_ruta, col_tiros_promedio_ruta,
                         col_volumen_promedio_met, col_importe_promedio_met, col_distancia_promedio_met, col_tiros_promedio_met]:
            if nueva_col in df_recorridos.columns:
                df_recorridos[nueva_col] = df_recorridos[nueva_col].apply(limpiar_numerico)
            else:
                # No crear valores por defecto para las nuevas columnas, se manejarán en calcular_estadisticas_ruta
                print(f"ℹ️ Columna opcional {nueva_col} no encontrada - se usarán cálculos alternativos")

        # Función para obtener el tamaño del marcador basado en volumen (MEJORADA)
        def obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min=8, tamano_max=40):
            if vol_max == vol_min or volumen == 0:
                return tamano_min
            # Usar escala logarítmica para mejor diferenciación visual
            if volumen <= 0:
                return tamano_min
            log_vol = np.log1p(volumen)  # log(1 + volumen) para evitar log(0)
            log_min = np.log1p(vol_min)
            log_max = np.log1p(vol_max)
            if log_max == log_min:
                return tamano_min
            proporcion = (log_vol - log_min) / (log_max - log_min)
            return int(tamano_min + proporcion * (tamano_max - tamano_min))

        # Función para obtener color basado en importe (MEJORADA - escala amarillo-naranja-rojo)
        def obtener_color_importe(importe, imp_min, imp_max):
            if imp_max == imp_min or importe == 0:
                return '#808080'  # Gris para valores sin importe
            proporcion = (importe - imp_min) / (imp_max - imp_min)
            # Escala mejorada: amarillo (bajo) → naranja (medio) → rojo (alto)
            if proporcion <= 0.5:
                # De amarillo a naranja
                rojo = int(255 * (0.8 + proporcion * 0.4))  # 204 a 255
                verde = int(255 * (1.0 - proporcion * 0.4))  # 255 a 204
                azul = 0
            else:
                # De naranja a rojo
                rojo = 255
                verde = int(255 * (1.6 - proporcion * 1.6))  # 204 a 0
                azul = 0
            return f'#{rojo:02x}{verde:02x}{azul:02x}'

        # NUEVA FUNCIÓN: Calcular centroide de ruta
        def obtener_centroide_ruta(ruta_data):
            """Calcula el centroide del polígono de la ruta a partir de un DataFrame"""
            try:
                # Extraer coordenadas del DataFrame
                coordenadas = [(row[col_latitud], row[col_longitud]) for _, row in ruta_data.iterrows()]
                
                if len(coordenadas) >= 3:
                    # Crear puntos y calcular convex hull
                    puntos = [Point(lon, lat) for lat, lon in coordenadas]
                    multipoint = MultiPoint(puntos)
                    hull = multipoint.convex_hull
                    
                    if hull.geom_type == 'Polygon':
                        centroide = hull.centroid
                        return (centroide.y, centroide.x)  # (lat, lon)
                    else:
                        # Fallback al promedio si no es polígono
                        lat_promedio = sum(lat for lat, lon in coordenadas) / len(coordenadas)
                        lon_promedio = sum(lon for lat, lon in coordenadas) / len(coordenadas)
                        return (lat_promedio, lon_promedio)
                else:
                    # Si hay menos de 3 puntos, usar el promedio
                    lat_promedio = sum(lat for lat, lon in coordenadas) / len(coordenadas)
                    lon_promedio = sum(lon for lat, lon in coordenadas) / len(coordenadas)
                    return (lat_promedio, lon_promedio)
            except Exception as e:
                # Fallback en caso de error - usar promedio simple
                try:
                    lat_promedio = ruta_data[col_latitud].mean()
                    lon_promedio = ruta_data[col_longitud].mean()
                    return (lat_promedio, lon_promedio)
                except:
                    return None

        # NUEVA FUNCIÓN: Calcular estadísticas por ruta
        def calcular_estadisticas_ruta(ruta_data):
            """Calcula estadísticas agregadas por ruta"""
            clientes_ruta = ruta_data[ruta_data[col_tipo] == 'Cliente']
            
            if clientes_ruta.empty:
                return None
                
            # Estadísticas básicas
            total_clientes = len(clientes_ruta)
            volumen_total = clientes_ruta[col_volumen].sum()
            importe_total = clientes_ruta[col_importe].sum()
            tiros_total = clientes_ruta[col_tiros].sum()
            distancia_total = clientes_ruta[col_distancia_sig].sum()
            
            # Promedios calculados
            volumen_promedio_calc = volumen_total / total_clientes if total_clientes > 0 else 0
            importe_promedio_calc = importe_total / total_clientes if total_clientes > 0 else 0
            tiros_promedio_calc = tiros_total / total_clientes if total_clientes > 0 else 0
            distancia_promedio_calc = distancia_total / total_clientes if total_clientes > 0 else 0
            
            # USAR NUEVAS COLUMNAS SI ESTÁN DISPONIBLES
            # Volumen: promedio, mín y máx por ruta desde las nuevas columnas
            if col_volumen_promedio_ruta in df_recorridos.columns and not ruta_data[col_volumen_promedio_ruta].isna().all():
                volumen_promedio_ruta = ruta_data[col_volumen_promedio_ruta].iloc[0]
            else:
                volumen_promedio_ruta = volumen_promedio_calc
                
            if col_volumen_min_ruta in df_recorridos.columns and not ruta_data[col_volumen_min_ruta].isna().all():
                volumen_min_ruta = ruta_data[col_volumen_min_ruta].iloc[0]
            else:
                volumen_min_ruta = clientes_ruta[col_volumen].min() if not clientes_ruta.empty else 0
                
            if col_volumen_max_ruta in df_recorridos.columns and not ruta_data[col_volumen_max_ruta].isna().all():
                volumen_max_ruta = ruta_data[col_volumen_max_ruta].iloc[0]
            else:
                volumen_max_ruta = clientes_ruta[col_volumen].max() if not clientes_ruta.empty else 0
            
            # Importe: promedio por ruta desde la nueva columna
            if col_importe_promedio_ruta in df_recorridos.columns and not ruta_data[col_importe_promedio_ruta].isna().all():
                importe_promedio_ruta = ruta_data[col_importe_promedio_ruta].iloc[0]
            else:
                importe_promedio_ruta = importe_promedio_calc
                
            # Distancia: promedio por ruta desde la nueva columna
            if col_distancia_promedio_ruta in df_recorridos.columns and not ruta_data[col_distancia_promedio_ruta].isna().all():
                distancia_promedio_ruta = ruta_data[col_distancia_promedio_ruta].iloc[0]
            else:
                distancia_promedio_ruta = distancia_promedio_calc
            
            # Tiros: promedio por ruta desde la nueva columna
            if col_tiros_promedio_ruta in df_recorridos.columns and not ruta_data[col_tiros_promedio_ruta].isna().all():
                tiros_promedio_ruta = ruta_data[col_tiros_promedio_ruta].iloc[0]
            else:
                tiros_promedio_ruta = tiros_promedio_calc
            
            # Top clientes
            cliente_max_vol = clientes_ruta.loc[clientes_ruta[col_volumen].idxmax()] if not clientes_ruta.empty else None
            cliente_max_imp = clientes_ruta.loc[clientes_ruta[col_importe].idxmax()] if not clientes_ruta.empty else None
            
            # Eficiencia
            eficiencia_vol_dist = volumen_total / max(distancia_total, 1)
            eficiencia_imp_dist = importe_total / max(distancia_total, 1)
            
            # Score de rentabilidad basado en VOLUMEN PROMEDIO y DISTANCIA PROMEDIO (más robusto ante outliers)
            score_rentabilidad = volumen_promedio_ruta / max(distancia_promedio_ruta, 0.1)
            
            # Categorización de clientes
            def calcular_categoria_cliente(vol_percentil, imp_percentil):
                if vol_percentil >= 80 and imp_percentil >= 80:
                    return "VIP"
                elif vol_percentil >= 70 or imp_percentil >= 70:
                    return "Premium"
                elif vol_percentil >= 50 or imp_percentil >= 50:
                    return "Estándar"
                else:
                    return "Básico"
            
            # Calcular percentiles dentro de la ruta
            if len(clientes_ruta) > 1:
                clientes_ruta_copy = clientes_ruta.copy()
                clientes_ruta_copy['percentil_vol'] = clientes_ruta_copy[col_volumen].rank(pct=True) * 100
                clientes_ruta_copy['percentil_imp'] = clientes_ruta_copy[col_importe].rank(pct=True) * 100
                clientes_ruta_copy['categoria'] = clientes_ruta_copy.apply(
                    lambda row: calcular_categoria_cliente(row['percentil_vol'], row['percentil_imp']), axis=1
                )
                
                categorias = clientes_ruta_copy['categoria'].value_counts()
                clientes_vip = categorias.get('VIP', 0)
                clientes_premium = categorias.get('Premium', 0)
                clientes_estandar = categorias.get('Estándar', 0)
                clientes_basico = categorias.get('Básico', 0)
            else:
                clientes_vip = clientes_premium = clientes_estandar = clientes_basico = 0
            
            return {
                'total_clientes': total_clientes,
                'volumen_total': volumen_total,
                'importe_total': importe_total,
                'tiros_total': tiros_total,
                'distancia_total': distancia_total,
                'volumen_promedio': volumen_promedio_calc,
                'importe_promedio': importe_promedio_calc,
                'tiros_promedio': tiros_promedio_calc,
                'distancia_promedio': distancia_promedio_calc,
                # NUEVAS MÉTRICAS DE LAS COLUMNAS ADICIONALES
                'volumen_promedio_ruta': volumen_promedio_ruta,
                'volumen_min_ruta': volumen_min_ruta,
                'volumen_max_ruta': volumen_max_ruta,
                'importe_promedio_ruta': importe_promedio_ruta,
                'tiros_promedio_ruta': tiros_promedio_ruta,
                'distancia_promedio_ruta': distancia_promedio_ruta,
                'cliente_max_vol': cliente_max_vol,
                'cliente_max_imp': cliente_max_imp,
                'eficiencia_vol_dist': eficiencia_vol_dist,
                'eficiencia_imp_dist': eficiencia_imp_dist,
                'score_rentabilidad': score_rentabilidad,
                'clientes_vip': clientes_vip,
                'clientes_premium': clientes_premium,
                'clientes_estandar': clientes_estandar,
                'clientes_basico': clientes_basico
            }

        # NUEVA FUNCIÓN: Normalizar scores de rentabilidad a escala 1-100
        def normalizar_scores_rentabilidad(scores_list):
            """Convierte una lista de scores de rentabilidad a escala 1-100"""
            if not scores_list or len(scores_list) == 0:
                return []
            
            import numpy as np
            scores_array = np.array(scores_list)
            
            # Evitar división por cero
            if scores_array.max() == scores_array.min():
                return [50] * len(scores_list)  # Valor medio si todos son iguales
            
            # Normalizar a escala 0-1 y luego a 1-100
            scores_norm = (scores_array - scores_array.min()) / (scores_array.max() - scores_array.min())
            scores_100 = 1 + (scores_norm * 99)  # Escala de 1 a 100
            
            return scores_100.tolist()

        # NOTA: Los rangos ahora se calculan dinámicamente dentro de generar_mapa() 
        # basándose en la selección de METs y rutas del usuario
        # 
        # NOTA: Los rangos ahora se calculan dinámicamente dentro de generar_mapa() 
        # basándose en la selección de METs y rutas del usuario
        # 
        # # Calcular rangos de volumen e importe para clientes
        # clientes_df = df_recorridos[df_recorridos[col_tipo] == 'Cliente']
        # if not clientes_df.empty:
        #     vol_min = clientes_df[col_volumen].min()
        #     vol_max = clientes_df[col_volumen].max()
        #     imp_min = clientes_df[col_importe].min()
        #     imp_max = clientes_df[col_importe].max()
        #     
        #     print(f"📊 Rangos de datos para clientes:")
        #     print(f"   Volumen: {vol_min:,.2f} - {vol_max:,.2f}")
        #     print(f"   Importe: ${imp_min:,.2f} - ${imp_max:,.2f}")
        #     print(f"   Total clientes: {len(clientes_df)}")
        # else:
        #     vol_min = vol_max = imp_min = imp_max = 0
        #     print("⚠️ No se encontraron registros de clientes")

        # Inicializar variables de rangos (serán calculadas dinámicamente)
        vol_min = vol_max = imp_min = imp_max = 0

        mets = sorted(df_recorridos[col_met].dropna().unique())
        rutas = sorted(df_recorridos[col_ruta].dropna().unique())
        
else:
    print("❌ No se puede continuar: DataFrame no válido o vacío")
    # Crear variables vacías para evitar errores
    mets = []
    rutas = []
    vol_min = vol_max = imp_min = imp_max = 0

# Controles interactivos MEJORADOS
met_selector = widgets.SelectMultiple(
    options=mets, 
    value=tuple(mets), 
    description='📍 Selecciona MET(s):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='50%')
)

ruta_selector = widgets.SelectMultiple(
    options=rutas, 
    value=tuple(rutas), 
    description='🛣️ Selecciona Ruta(s):', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='50%')
)

todos_clientes_checkbox = widgets.Checkbox(
    value=True, 
    description='Procesar todos los clientes', 
    indent=False
)

todas_rutas_checkbox = widgets.Checkbox(
    value=True, 
    description='Seleccionar todas las rutas', 
    indent=False
)

clientes_a_procesar = widgets.IntText(
    value=10, 
    description='Cantidad de clientes:', 
    style={'description_width': 'initial'}, 
    layout=widgets.Layout(width='40%')
)

# CONTROLES PARA VISUALIZACIÓN SIMPLIFICADOS
tipo_visualizacion = widgets.Dropdown(
    options=[
        ('🎯 Vista por Rutas (Principal)', 'rutas_agrupadas'),
        ('🔥 Mapas de Calor', 'solo_heatmap'),
        ('🎨 Rutas + Mapas de Calor', 'completo')
    ],
    value='rutas_agrupadas',
    description='🎨 Visualización:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

# Casillas de mapas de calor removidas - ahora se controlan desde el dropdown principal

escala_marcadores = widgets.Dropdown(
    options=[
        ('🔹 Pequeños (6-25px)', 'small'),
        ('🔸 Medianos (8-35px)', 'medium'),
        ('🔶 Grandes (10-50px)', 'large')
    ],
    value='medium',
    description='📏 Tamaño marcadores:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

generar_btn = widgets.Button(
    description='🗺️ Generar Mapa de Análisis', 
    button_style='success',
    layout=widgets.Layout(width='300px', height='40px')
)

analisis_excel_btn = widgets.Button(
    description='📊 Generar Análisis Excel', 
    button_style='info',
    layout=widgets.Layout(width='300px', height='40px')
)

output_map = widgets.Output()
output_excel = widgets.Output()

# Actualiza rutas disponibles según MET seleccionado
def actualizar_rutas(*args):
    mets_seleccionados = list(met_selector.value)
    if mets_seleccionados:
        rutas_filtradas = sorted(df_recorridos[df_recorridos[col_met].isin(mets_seleccionados)][col_ruta].dropna().unique())
    else:
        rutas_filtradas = sorted(df_recorridos[col_ruta].dropna().unique())
    ruta_selector.options = rutas_filtradas
    rutas_validas = [r for r in ruta_selector.value if r in rutas_filtradas]
    if rutas_validas:
        ruta_selector.value = tuple(rutas_validas)
    elif rutas_filtradas:
        ruta_selector.value = (rutas_filtradas[0],)
    else:
        ruta_selector.value = ()

# Función para controlar selección de todas las rutas
def controlar_todas_rutas(*args):
    if todas_rutas_checkbox.value:
        # Seleccionar todas las rutas disponibles
        ruta_selector.value = tuple(ruta_selector.options)
    else:
        # Deseleccionar todas las rutas
        ruta_selector.value = ()

# Función para actualizar el checkbox cuando se cambian las rutas manualmente
def actualizar_checkbox_rutas(*args):
    # Si todas las rutas están seleccionadas, marcar el checkbox
    todas_rutas_checkbox.value = len(ruta_selector.value) == len(ruta_selector.options) and len(ruta_selector.options) > 0

met_selector.observe(actualizar_rutas, names='value')
todas_rutas_checkbox.observe(controlar_todas_rutas, names='value')
ruta_selector.observe(actualizar_checkbox_rutas, names='value')
actualizar_rutas()

# Paleta de colores para MET General
colores_met_general = ['#FF6B00', '#00529B', '#FFA500', '#8BC34A', '#E91E63', '#00BCD4', '#FF9800', '#9C27B0', '#607D8B', '#CDDC39', '#795548', '#3F51B5', '#F44336', '#009688', '#C0CA33', '#7B1FA2', '#D32F2F', '#388E3C', '#1976D2', '#FFA07A']

def generar_mapa(b):
    with output_map:
        clear_output()
        
        # Verificar que los datos estén disponibles
        if df_recorridos is None or df_recorridos.empty:
            print('❌ No hay datos válidos para generar el mapa.')
            print('Por favor, verifica que el archivo Excel contenga las columnas necesarias.')
            return
            
        mets_seleccionados = list(met_selector.value)
        rutas_seleccionadas = list(ruta_selector.value)
        if not mets_seleccionados:
            print('Selecciona al menos un MET para generar el mapa.')
            return
        if not rutas_seleccionadas:
            print('Selecciona al menos una ruta para generar el mapa.')
            return

        print(f"🚀 Generando mapa con visualización '{tipo_visualizacion.value}'...")
        
        # CALCULAR RANGOS DINÁMICAMENTE basándose en la selección del usuario
        clientes_seleccionados = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas)) & 
            (df_recorridos[col_tipo] == 'Cliente')
        ]
        
        if not clientes_seleccionados.empty:
            vol_min = clientes_seleccionados[col_volumen].min()
            vol_max = clientes_seleccionados[col_volumen].max()
            imp_min = clientes_seleccionados[col_importe].min()
            imp_max = clientes_seleccionados[col_importe].max()
        else:
            vol_min = vol_max = imp_min = imp_max = 0
        
        met_row = df_recorridos[df_recorridos[col_met] == mets_seleccionados[0]].iloc[0]
        mapa = folium.Map(location=[met_row[col_latitud], met_row[col_longitud]], zoom_start=10, tiles='OpenStreetMap')
        
        # Obtener configuración de tamaños
        tamanos_config = {
            'small': (6, 25),
            'medium': (8, 35),
            'large': (10, 50)
        }
        tamano_min, tamano_max = tamanos_config[escala_marcadores.value]
        
        # Preparar datos para mapas de calor
        clientes_todos = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas)) & 
            (df_recorridos[col_tipo] == 'Cliente')
        ]
        
        # Datos para mapas de calor
        heat_data_volumen = []
        heat_data_importe = []
        for _, row in clientes_todos.iterrows():
            lat, lon = row[col_latitud], row[col_longitud]
            vol = row[col_volumen] if col_volumen in df_recorridos.columns and not pd.isna(row[col_volumen]) else 0
            imp = row[col_importe] if col_importe in df_recorridos.columns and not pd.isna(row[col_importe]) else 0
            if vol > 0:
                heat_data_volumen.append([lat, lon, vol])
            if imp > 0:
                heat_data_importe.append([lat, lon, imp])
        
        # Añadir mapas de calor si se solicitan
        if tipo_visualizacion.value in ['solo_heatmap', 'completo']:
            if heat_data_volumen:
                HeatMap(
                    heat_data_volumen, 
                    name='🔥 Mapa de Calor - Volumen', 
                    radius=25, 
                    blur=15, 
                    max_zoom=1,
                    gradient={0.2: 'yellow', 0.4: 'orange', 0.6: 'darkorange', 0.8: 'red', 1: 'darkred'},
                    show=False
                ).add_to(mapa)
        
        if tipo_visualizacion.value in ['solo_heatmap', 'completo']:
            if heat_data_importe:
                HeatMap(
                    heat_data_importe, 
                    name='💰 Mapa de Calor - Importe', 
                    radius=25, 
                    blur=15, 
                    max_zoom=1,
                    gradient={0.2: 'yellow', 0.4: 'orange', 0.6: 'darkorange', 0.8: 'red', 1: 'darkred'},
                    show=False
                ).add_to(mapa)

        featuregroups_combo = {}
        colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray']

        resumen_combos = {}
        resumen_mets = {}
        color_idx = 0
        met_general_idx = 0
        
        # Crear una capa por cada combinación MET-Ruta
        for met in mets_seleccionados:
            met_clean = str(met).strip()
            if met_clean.startswith('MET '):
                met_clean = met_clean[4:]
            met_id = met_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
            met_recorridos = df_recorridos[(df_recorridos[col_met] == met) & (df_recorridos[col_ruta].isin(rutas_seleccionadas))]
            rutas_met = met_recorridos[col_ruta].unique()
            
            for ruta in rutas_met:
                ruta_clean = str(ruta).strip()
                ruta_id = ruta_clean.replace(' ', '').replace(':', '').replace('.', '').replace('é', 'e').replace('á', 'a').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
                combo_id = f"{met_id}-{ruta_id}"
                capa_nombre = f"MET {met_clean} - Ruta {ruta_clean}"
                fg_combo = folium.FeatureGroup(name=capa_nombre, show=True)
                featuregroups_combo[combo_id] = fg_combo
                ruta_recorridos = met_recorridos[met_recorridos[col_ruta] == ruta]
                
                if ruta_recorridos.empty:
                    continue
                    
                ruta_recorridos_sorted = ruta_recorridos.sort_values(col_secuencia)
                coords = list(zip(ruta_recorridos_sorted[col_latitud], ruta_recorridos_sorted[col_longitud]))
                color_ruta = colores[color_idx % len(colores)]
                color_idx += 1
                
                # Añadir línea de ruta solo si NO es rutas agrupadas ni solo mapa de calor
                if tipo_visualizacion.value not in ['rutas_agrupadas', 'solo_heatmap']:
                    folium.PolyLine(
                        coords, 
                        color=color_ruta, 
                        weight=5, 
                        opacity=0.8, 
                        tooltip=capa_nombre,
                        className=f'ruta-line ruta-{combo_id}',
                        **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                    ).add_to(fg_combo)
                
                # Crear polígono del área de ruta (convex hull de todos los puntos)
                puntos_ruta = [Point(row[col_longitud], row[col_latitud]) for _, row in ruta_recorridos_sorted.iterrows()]
                if len(puntos_ruta) >= 3:
                    multipoint = MultiPoint(puntos_ruta)
                    hull = multipoint.convex_hull
                    if hull.geom_type == 'Polygon':
                        coords_polygon = [(lat, lon) for lon, lat in hull.exterior.coords]
                        area_popup = f'''
                        <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                            <div style="font-size: 14px; font-weight: bold;">📍 Área de Cobertura</div>
                            <div style="font-size: 12px; margin-top: 4px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</div>
                            <div style="font-size: 11px; margin-top: 2px;">👥 {ruta_recorridos[ruta_recorridos[col_tipo] == 'Cliente'].shape[0]} clientes</div>
                        </div>
                        '''
                        # Solo mostrar polígono si NO es rutas agrupadas
                        if tipo_visualizacion.value != 'rutas_agrupadas':
                            folium.Polygon(
                                locations=coords_polygon,
                                color=color_ruta,
                                weight=2,
                                opacity=0.7,
                                fillColor=color_ruta,
                                fillOpacity=0.15,
                                popup=folium.Popup(area_popup, max_width=250),
                                className=f'area-ruta area-{combo_id}',
                                **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                            ).add_to(fg_combo)
                
                # NUEVA FUNCIONALIDAD: Polígono SOLO de clientes (sin METs)
                clientes_ruta = ruta_recorridos_sorted[ruta_recorridos_sorted[col_tipo] == 'Cliente']
                if len(clientes_ruta) >= 3:
                    puntos_solo_clientes = [Point(row[col_longitud], row[col_latitud]) for _, row in clientes_ruta.iterrows()]
                    if len(puntos_solo_clientes) >= 3:
                        multipoint_clientes = MultiPoint(puntos_solo_clientes)
                        hull_clientes = multipoint_clientes.convex_hull
                        if hull_clientes.geom_type == 'Polygon':
                            coords_polygon_clientes = [(lat, lon) for lon, lat in hull_clientes.exterior.coords]
                            area_popup_clientes = f'''
                            <div style="background: {color_ruta}; color: white; padding: 8px; border-radius: 8px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif;">
                                <div style="font-size: 14px; font-weight: bold;">👥 Área Solo Clientes</div>
                                <div style="font-size: 12px; margin-top: 4px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</div>
                                <div style="font-size: 11px; margin-top: 2px;">👨‍💼 {len(clientes_ruta)} clientes</div>
                                <div style="font-size: 10px; margin-top: 2px; opacity: 0.9;">📦 Vol Promedio: {clientes_ruta[col_volumen].mean():,.1f} m³</div>
                            </div>
                            '''
                            # Solo mostrar polígono si NO es rutas agrupadas
                            if tipo_visualizacion.value != 'rutas_agrupadas':
                                folium.Polygon(
                                    locations=coords_polygon_clientes,
                                    color=color_ruta,
                                    weight=2,
                                    opacity=0.9,
                                    fillColor=color_ruta,
                                    fillOpacity=0.1,
                                    popup=folium.Popup(area_popup_clientes, max_width=250),
                                    className=f'area-clientes area-clientes-{combo_id}',
                                    **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                                ).add_to(fg_combo)
                
                # 🚀 NUEVA FUNCIONALIDAD: ESFERA POR RUTA (RUTAS AGRUPADAS)
                if tipo_visualizacion.value == 'rutas_agrupadas':
                    # Calcular estadísticas de la ruta
                    stats_ruta = calcular_estadisticas_ruta(ruta_recorridos)
                    
                    # Obtener centroide de la ruta
                    centroide = obtener_centroide_ruta(ruta_recorridos)
                    
                    if centroide:
                        lat_centroide, lon_centroide = centroide
                        
                        # Calcular tamaño de la esfera basado en VOLUMEN PROMEDIO (consistente con los datos mostrados)
                        vol_promedio = stats_ruta['volumen_promedio_ruta']
                        
                        # MEJORADO: Calcular rangos dinámicos de volumen PROMEDIO para todas las rutas seleccionadas
                        # Este algoritmo hace que el tamaño de las esferas sea proporcional al volumen promedio:
                        # - Encuentra el volumen promedio mínimo y máximo entre todas las rutas seleccionadas
                        # - Calcula una proporción (0-1) basada en el volumen promedio de la ruta actual
                        # - Mapea esa proporción a un rango de tamaños (20-80 píxeles)
                        # - Las rutas con más volumen promedio tendrán esferas más grandes
                        todos_volumenes_promedios = []
                        for met_temp in mets_seleccionados:
                            met_recorridos_temp = df_recorridos[df_recorridos[col_met] == met_temp]
                            rutas_temp = met_recorridos_temp[met_recorridos_temp[col_ruta].isin(rutas_seleccionadas)][col_ruta].unique()
                            for ruta_temp in rutas_temp:
                                ruta_temp_data = met_recorridos_temp[met_recorridos_temp[col_ruta] == ruta_temp]
                                stats_temp = calcular_estadisticas_ruta(ruta_temp_data)
                                if stats_temp:
                                    todos_volumenes_promedios.append(stats_temp['volumen_promedio_ruta'])
                        
                        if todos_volumenes_promedios:
                            vol_prom_min_rutas = min(todos_volumenes_promedios)
                            vol_prom_max_rutas = max(todos_volumenes_promedios)
                            
                            # Calcular tamaño proporcional al volumen promedio (rango 20-80 píxeles)
                            if vol_prom_max_rutas > vol_prom_min_rutas:
                                proporcion_vol = (vol_promedio - vol_prom_min_rutas) / (vol_prom_max_rutas - vol_prom_min_rutas)
                                tamano_esfera = 20 + (proporcion_vol * 60)  # Rango 20-80 píxeles
                            else:
                                tamano_esfera = 50  # Tamaño por defecto si todos tienen el mismo volumen
                        else:
                            tamano_esfera = 30  # Tamaño por defecto si no hay datos
                        
                        # Asegurar que el tamaño esté en un rango razonable
                        tamano_esfera = max(20, min(80, tamano_esfera))
                        
                        # NUEVO: Calcular rangos dinámicos basados en todos los scores de la selección actual
                        todos_scores = []
                        rutas_info = []  # Para guardar info de cada ruta
                        for met_temp in mets_seleccionados:
                            met_recorridos_temp = df_recorridos[df_recorridos[col_met] == met_temp]
                            rutas_temp = met_recorridos_temp[met_recorridos_temp[col_ruta].isin(rutas_seleccionadas)][col_ruta].unique()
                            for ruta_temp in rutas_temp:
                                ruta_temp_data = met_recorridos_temp[met_recorridos_temp[col_ruta] == ruta_temp]
                                stats_temp = calcular_estadisticas_ruta(ruta_temp_data)
                                if stats_temp:
                                    todos_scores.append(stats_temp['score_rentabilidad'])
                                    rutas_info.append((met_temp, ruta_temp, stats_temp))
                        
                        # Normalizar todos los scores a escala 1-100
                        scores_normalizados = normalizar_scores_rentabilidad(todos_scores)
                        
                        # Encontrar el score normalizado para la ruta actual
                        ruta_actual_idx = None
                        for i, (met_temp, ruta_temp, stats_temp) in enumerate(rutas_info):
                            if met_temp == met and ruta_temp == ruta:
                                ruta_actual_idx = i
                                break
                        
                        # Asignar score normalizado
                        if ruta_actual_idx is not None:
                            score_normalizado = scores_normalizados[ruta_actual_idx]
                        else:
                            score_normalizado = 50  # Valor por defecto
                        
                        # Usar percentiles fijos para categorización con scores normalizados
                        if score_normalizado >= 50:
                            color_esfera = '#44AA44'  # Verde - Muy eficiente (>50)
                            categoria_eficiencia =f"🟢 Muy Eficiente (Score: {score_normalizado:.1f}/100)"
                        elif score_normalizado >= 30:
                            color_esfera = '#FFBB00'  # Amarillo - Eficiente (30-50)
                            categoria_eficiencia =f"🟡 Eficiente (Score: {score_normalizado:.1f}/100)"
                        elif score_normalizado >= 20:
                            color_esfera = '#FF8800'  # Naranja - Moderado (20-30)
                            categoria_eficiencia =f"🟠 Moderado (Score: {score_normalizado:.1f}/100)"
                        else:
                            color_esfera = '#FF4444'  # Rojo - Bajo rendimiento (<20)
                            categoria_eficiencia =f"🔴 Bajo Rendimiento (Score: {score_normalizado:.1f}/100)"
                        
                        # 🆕 CALCULAR DATOS DE OTROS METs PARA ESTA MISMA RUTA
                        otros_mets_data = []
                        for met_temp in mets_seleccionados:
                            if met_temp != met:  # Solo otros METs, no el actual
                                met_temp_recorridos = df_recorridos[df_recorridos[col_met] == met_temp]
                                ruta_temp_data = met_temp_recorridos[met_temp_recorridos[col_ruta] == ruta]
                                if not ruta_temp_data.empty:
                                    stats_temp = calcular_estadisticas_ruta(ruta_temp_data)
                                    if stats_temp:
                                        met_temp_clean = str(met_temp).strip()
                                        if met_temp_clean.startswith('MET '):
                                            met_temp_clean = met_temp_clean[4:]
                                        otros_mets_data.append({
                                            'met': met_temp_clean,
                                            'volumen_promedio': stats_temp['volumen_promedio_ruta'],
                                            'importe_promedio': stats_temp['importe_promedio_ruta'],
                                            'distancia_promedio': stats_temp['distancia_promedio_ruta'],
                                            'tiros_promedio': stats_temp['tiros_promedio_ruta'],
                                            'total_clientes': stats_temp['total_clientes']
                                        })
                        
                        # Construir HTML para otros METs (si existen)
                        otros_mets_html = ""
                        if otros_mets_data:
                            otros_mets_html = f'''
                            <div style="background: linear-gradient(135deg, #E8EAF6 0%, #C5CAE9 100%); padding: 14px; border-radius: 12px; margin-bottom: 12px; border: 2px solid #3F51B5;">
                                <div style="font-size: 13px; font-weight: bold; color: #1A237E; margin-bottom: 10px; text-align: center;">
                                    📊 OTROS METs EN RUTA {ruta_clean} ({len(otros_mets_data)} MET{'s' if len(otros_mets_data) > 1 else ''})
                                </div>
                            '''
                            for idx, datos_met in enumerate(otros_mets_data):
                                otros_mets_html += f'''
                                <div style="background: white; padding: 10px; border-radius: 8px; margin-bottom: {'8px' if idx < len(otros_mets_data)-1 else '0'}; border-left: 4px solid #3F51B5;">
                                    <div style="font-size: 11px; font-weight: bold; color: #3F51B5; margin-bottom: 6px;">🏠 MET {datos_met['met']} ({datos_met['total_clientes']} clientes)</div>
                                    <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 6px; font-size: 10px;">
                                        <div style="text-align: center;">
                                            <div style="color: #2E7D32; font-weight: bold;">📦 {datos_met['volumen_promedio']:,.2f} m³</div>
                                        </div>
                                        <div style="text-align: center;">
                                            <div style="color: #1976D2; font-weight: bold;">💰 ${datos_met['importe_promedio']:,.0f}</div>
                                        </div>
                                        <div style="text-align: center;">
                                            <div style="color: #F57C00; font-weight: bold;">📏 {datos_met['distancia_promedio']:,.1f} km</div>
                                        </div>
                                        <div style="text-align: center;">
                                            <div style="color: #7B1FA2; font-weight: bold;">🎯 {datos_met['tiros_promedio']:,.1f}</div>
                                        </div>
                                    </div>
                                </div>
                                '''
                            otros_mets_html += '''
                            </div>
                            '''
                        
                        # Popup con información agregada de la ruta (SOLO PROMEDIOS)
                        popup_ruta = f'''
                        <div style="background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%); border-radius: 16px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid {color_esfera}; min-width: 380px; max-width: 450px; font-family: 'Segoe UI', Arial, sans-serif;">
                            <div style="background: {color_esfera}; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 13px 13px 8px 8px; text-align: center;">
                                <div style="font-size: 20px; font-weight: bold;">🛣️ RUTA {ruta_clean}</div>
                                <div style="font-size: 14px; opacity: 0.95;">🏠 MET {met_clean}</div>
                            </div>
                            
                            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin-bottom: 16px;">
                                <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #4CAF50;">
                                    <div style="font-size: 11px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{stats_ruta['volumen_promedio_ruta']:,.3f} m³</div>
                                    <div style="font-size: 9px; color: #2E7D32;">Min: {stats_ruta['volumen_min_ruta']:,.3f} | Max: {stats_ruta['volumen_max_ruta']:,.3f}</div>
                                </div>
                                <div style="background: #E3F2FD; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #2196F3;">
                                    <div style="font-size: 11px; color: #1976D2; font-weight: bold;">💰 IMPORTE PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #0D47A1; margin: 4px 0;">${stats_ruta['importe_promedio_ruta']:,.0f}</div>
                                </div>
                                <div style="background: #FFF3E0; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #FF9800;">
                                    <div style="font-size: 11px; color: #F57C00; font-weight: bold;">📏 DISTANCIA PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #E65100; margin: 4px 0;">{stats_ruta['distancia_promedio_ruta']:,.1f} km</div>
                                </div>
                                <div style="background: #F3E5F5; padding: 12px; border-radius: 12px; text-align: center; border: 2px solid #9C27B0;">
                                    <div style="font-size: 11px; color: #7B1FA2; font-weight: bold;">🎯 TIROS PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #4A148C; margin: 4px 0;">{stats_ruta['tiros_promedio_ruta']:,.1f}</div>
                                </div>
                            </div>
                            
                            <div style="background: #E8F5E8; padding: 12px; border-radius: 12px; margin-bottom: 12px;">
                                <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 6px;">⚡ EFICIENCIA</div>
                                <div style="background: {color_esfera}; color: white; padding: 6px 10px; border-radius: 15px; text-align: center; margin-bottom: 8px; font-size: 11px; font-weight: bold;">
                                    {categoria_eficiencia}
                                </div>
                            </div>
                            
                            {otros_mets_html}
                            
                            <div style="font-size: 10px; color: #666; text-align: center; border-top: 1px solid #ddd; padding-top: 8px;">
                                🎯 Vista Rutas Agrupadas - Datos promedio por ruta<br>
                                📍 Centroide geográfico | 🎨 Color: Eficiencia | 📏 Tamaño: Volumen promedio
                            </div>
                        </div>
                        '''
                        
                        # Crear esfera principal para la ruta
                        folium.CircleMarker(
                            location=[lat_centroide, lon_centroide],
                            popup=folium.Popup(popup_ruta, max_width=480),
                            radius=tamano_esfera,
                            color='#FFFFFF',
                            weight=4,
                            fillColor=color_esfera,
                            fillOpacity=0.8,
                            className=f'ruta-esfera ruta-{combo_id}',
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                        
                        # Marcador con icono de ruta en el centro
                        folium.Marker(
                            location=[lat_centroide, lon_centroide],
                            icon=DivIcon(
                                icon_size=(32, 32),
                                icon_anchor=(16, 16),
                                html=f'''
                                <div style="
                                    width: 32px; 
                                    height: 32px; 
                                    background: {color_esfera}; 
                                    border: 3px solid white; 
                                    border-radius: 50%; 
                                    display: flex; 
                                    align-items: center; 
                                    justify-content: center; 
                                    font-size: 12px; 
                                    color: white; 
                                    font-weight: bold;
                                    box-shadow: 0 4px 8px rgba(0,0,0,0.3);
                                ">{ruta_clean}</div>
                                '''
                            ),
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                        
                        # CREAR ÁREA DE COBERTURA PARA RUTAS AGRUPADAS
                        puntos_ruta = [Point(row[col_longitud], row[col_latitud]) for _, row in ruta_recorridos_sorted.iterrows()]
                        if len(puntos_ruta) >= 3:
                            multipoint = MultiPoint(puntos_ruta)
                            hull = multipoint.convex_hull
                            if hull.geom_type == 'Polygon':
                                coords_polygon = [(lat, lon) for lon, lat in hull.exterior.coords]
                                area_popup = f'''
                                <div style="background: {color_esfera}; color: white; padding: 12px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif; min-width: 280px;">
                                    <div style="font-size: 16px; font-weight: bold; margin-bottom: 8px;">📍 Área de Cobertura</div>
                                    <div style="font-size: 14px; margin-bottom: 4px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</div>
                                    <div style="font-size: 12px; margin-bottom: 8px;">👥 {stats_ruta['total_clientes']} clientes | 📏 {stats_ruta['distancia_promedio_ruta']:,.1f} km promedio</div>
                                    <div style="background: rgba(255,255,255,0.2); padding: 8px; border-radius: 8px;">
                                        <div style="font-size: 11px;">📦 Vol: {stats_ruta['volumen_promedio_ruta']:,.1f} m³ | 💰 Imp: ${stats_ruta['importe_promedio_ruta']:,.0f}</div>
                                        <div style="font-size: 11px; margin-top: 4px;">{categoria_eficiencia}</div>
                                    </div>
                                </div>
                                '''
                                folium.Polygon(
                                    locations=coords_polygon,
                                    color=color_esfera,
                                    weight=3,
                                    opacity=0.8,
                                    fillColor=color_esfera,
                                    fillOpacity=0.25,
                                    popup=folium.Popup(area_popup, max_width=300),
                                    className=f'area-ruta area-{combo_id}',
                                    **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                                ).add_to(fg_combo)
                        
                        # NUEVA FUNCIONALIDAD: Polígono SOLO de clientes para rutas agrupadas (sin METs)
                        clientes_ruta_agrupada = ruta_recorridos_sorted[ruta_recorridos_sorted[col_tipo] == 'Cliente']
                        if len(clientes_ruta_agrupada) >= 3:
                            puntos_solo_clientes_agrupada = [Point(row[col_longitud], row[col_latitud]) for _, row in clientes_ruta_agrupada.iterrows()]
                            if len(puntos_solo_clientes_agrupada) >= 3:
                                multipoint_clientes_agrupada = MultiPoint(puntos_solo_clientes_agrupada)
                                hull_clientes_agrupada = multipoint_clientes_agrupada.convex_hull
                                if hull_clientes_agrupada.geom_type == 'Polygon':
                                    coords_polygon_clientes_agrupada = [(lat, lon) for lon, lat in hull_clientes_agrupada.exterior.coords]
                                    area_popup_clientes_agrupada = f'''
                                    <div style="background: {color_esfera}; color: white; padding: 12px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', Arial, sans-serif; min-width: 280px;">
                                        <div style="font-size: 16px; font-weight: bold; margin-bottom: 8px;">👥 Área Solo Clientes</div>
                                        <div style="font-size: 14px; margin-bottom: 4px;">🏠 MET {met_clean} - 🛣️ Ruta {ruta_clean}</div>
                                        <div style="font-size: 12px; margin-bottom: 8px;">👨‍💼 {len(clientes_ruta_agrupada)} clientes</div>
                                        <div style="background: rgba(255,255,255,0.2); padding: 8px; border-radius: 8px;">
                                            <div style="font-size: 11px;">📦 Vol Promedio: {clientes_ruta_agrupada[col_volumen].mean():,.1f} m³ | 💰 Imp Promedio: ${clientes_ruta_agrupada[col_importe].mean():,.0f}</div>
                                            <div style="font-size: 11px; margin-top: 4px;">{categoria_eficiencia}</div>
                                        </div>
                                    </div>
                                    '''
                                    folium.Polygon(
                                        locations=coords_polygon_clientes_agrupada,
                                        color=color_esfera,
                                        weight=2,
                                        opacity=0.9,
                                        fillColor=color_esfera,
                                        fillOpacity=0.1,
                                        popup=folium.Popup(area_popup_clientes_agrupada, max_width=300),
                                        className=f'area-clientes area-clientes-{combo_id}',
                                        **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                                    ).add_to(fg_combo)
                
                # PROCESAMIENTO DE MARCADORES INDIVIDUALES (Solo si NO es rutas agrupadas)
                if tipo_visualizacion.value != 'rutas_agrupadas':
                    for i, row in ruta_recorridos_sorted.iterrows():
                        # Marcadores para clientes (MEJORADOS)
                        if row[col_tipo] == 'Cliente' and tipo_visualizacion.value != 'solo_heatmap':
                            volumen = row[col_volumen] if col_volumen in df_recorridos.columns else 0
                            importe = row[col_importe] if col_importe in df_recorridos.columns else 0
                            tiros = row[col_tiros] if col_tiros in df_recorridos.columns else 0
                            tamano_marcador = obtener_tamano_marcador(volumen, vol_min, vol_max, tamano_min, tamano_max)
                            color_importe = obtener_color_importe(importe, imp_min, imp_max)
                            
                            # Calcular percentiles para mostrar en el popup
                            percentil_vol = ((volumen - vol_min) / (vol_max - vol_min) * 100) if vol_max > vol_min else 0
                            percentil_imp = ((importe - imp_min) / (imp_max - imp_min) * 100) if imp_max > imp_min else 0
                        
                            # Determinar categoría de cliente
                            if percentil_vol >= 80 and percentil_imp >= 80:
                                categoria = "⭐ VIP (Alto Volumen + Alto Importe)"
                                categoria_color = "#FFD700"
                            elif percentil_vol >= 70 or percentil_imp >= 70:
                                categoria = "🔸 Premium (Alto en una métrica)"
                                categoria_color = "#FF8C00"
                            elif percentil_vol >= 50 or percentil_imp >= 50:
                                categoria = "🔹 Estándar"
                                categoria_color = "#4682B4"
                            else:
                                categoria = "⚪ Básico"
                                categoria_color = "#808080"
                        
                            if tipo_visualizacion.value == 'marcadores_smart':
                                popup_html = f'''
                                <div style="background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%); border-radius: 16px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 16px; border: 3px solid {color_importe}; min-width: 320px; max-width: 400px; font-family: 'Segoe UI', Arial, sans-serif;">
                                    <div style="background: {color_importe}; color: white; margin: -16px -16px 12px -16px; padding: 12px; border-radius: 13px 13px 8px 8px; text-align: center;">
                                        <div style="font-size:18px; font-weight:bold;">👨‍💼 {row[col_codigo]}</div>
                                        <div style="font-size:12px; opacity:0.9;">{row[col_nombre]}</div>
                                    </div>
                                    <div style="background: {categoria_color}; color: white; padding: 6px 10px; border-radius: 20px; text-align: center; margin-bottom: 12px; font-size: 12px; font-weight: bold;">
                                        {categoria}
                                    </div>
                                    <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 8px; margin-bottom: 12px;">
                                        <div style="background: #E8F5E8; padding: 8px; border-radius: 8px; text-align: center;">
                                            <div style="font-size: 11px; color: #2E7D32;">📦 VOLUMEN</div>
                                            <div style="font-size: 14px; font-weight: bold; color: #1B5E20;">{volumen:,.3f} m³</div>
                                            <div style="font-size: 10px; color: #388E3C;">Top {100-percentil_vol:.0f}°</div>
                                            <div style="background: #C8E6C9; height: 4px; border-radius: 2px; margin-top: 4px;">
                                                <div style="background: #4CAF50; height: 100%; width: {percentil_vol:.0f}%; border-radius: 2px;"></div>
                                            </div>
                                        </div>
                                        <div style="background: #E3F2FD; padding: 8px; border-radius: 8px; text-align: center;">
                                            <div style="font-size: 11px; color: #1976D2;">💰 IMPORTE</div>
                                            <div style="font-size: 14px; font-weight: bold; color: #0D47A1;">${importe:,.0f}</div>
                                            <div style="font-size: 10px; color: #1976D2;">Top {100-percentil_imp:.0f}°</div>
                                            <div style="background: #BBDEFB; height: 4px; border-radius: 2px; margin-top: 4px;">
                                                <div style="background: {color_importe}; height: 100%; width: {percentil_imp:.0f}%; border-radius: 2px;"></div>
                                            </div>
                                        </div>
                                        <div style="background: #F3E5F5; padding: 8px; border-radius: 8px; text-align: center;">
                                            <div style="font-size: 11px; color: #7B1FA2;">🎯 TIROS</div>
                                            <div style="font-size: 14px; font-weight: bold; color: #4A148C;">{tiros:,.1f}</div>
                                            <div style="font-size: 10px; color: #7B1FA2;">Info</div>
                                        </div>
                                    </div>
                                    <div style="font-size: 11px; color: #666; text-align: center; border-top: 1px solid #ddd; padding-top: 8px;">
                                        🗺️ Ruta: <b>{row[col_ruta]}</b> | 🔢 Pos: <b>{row[col_secuencia]}</b> | 📏 Dist: <b>{row[col_distancia_sig]} km</b>
                                    </div>
                                </div>
                                '''
                            else:
                                # Popup básico para marcadores normales
                                popup_html = f'''
                                <div style="background: #fff; border-radius: 12px; padding: 12px; border: 2px solid {color_importe}; min-width: 280px;">
                                    <div style="font-size:16px; color:{color_ruta}; font-weight:bold; margin-bottom:8px;">
                                        👨‍💼 {row[col_codigo]} - {row[col_nombre]}
                                    </div>
                                    <div style="font-size:14px; color:#FF6B00; margin-bottom:4px;">
                                        📦 Volumen: {volumen:,.3f} m³ (Top {100-percentil_vol:.0f}°)
                                    </div>
                                    <div style="font-size:14px; color:{color_importe}; margin-bottom:4px;">
                                        💰 Importe: ${importe:,.2f} (Top {100-percentil_imp:.0f}°)
                                    </div>
                                    <div style="font-size:14px; color:#7B1FA2; margin-bottom:4px;">
                                        🎯 Tiros: {tiros:,.1f}
                                    </div>
                                    <div style="font-size:12px; color:#666;">
                                        🗺️ {row[col_ruta]} | Pos: {row[col_secuencia]} | Dist: {row[col_distancia_sig]} km
                                    </div>
                                </div>
                                '''
                            
                            # Marcador principal con tamaño por volumen y color por importe
                            folium.CircleMarker(
                                location=[row[col_latitud], row[col_longitud]],
                                popup=folium.Popup(popup_html, max_width=420),
                                radius=tamano_marcador,
                                color='#333333',
                                weight=3,
                                fillColor=color_importe,
                                fillOpacity=0.85,
                                className=f'cliente-marker cliente-{combo_id}',
                                **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                            ).add_to(fg_combo)
                        # Marcadores para METs (SIMPLIFICADOS - solo estadísticas agregadas)
                        elif row[col_tipo] == 'MET' and tipo_visualizacion.value != 'solo_heatmap':
                            # Calcular estadísticas agregadas para este MET
                            clientes_met = clientes_todos[clientes_todos[col_met] == met]
                            if not clientes_met.empty:
                                volumen_total_met = clientes_met[col_volumen].sum()
                                importe_total_met = clientes_met[col_importe].sum()
                                tiros_total_met = clientes_met[col_tiros].sum()
                                distancia_total_met = clientes_met[col_distancia_sig].sum()
                                total_clientes_met = len(clientes_met)
                            else:
                                volumen_total_met = importe_total_met = tiros_total_met = distancia_total_met = total_clientes_met = 0
                            
                            # OBTENER DATOS DIRECTOS DE MET DESDE LA BASE DE DATOS
                            volumen_promedio_met_bd = row.get(col_volumen_promedio_met, 0) if col_volumen_promedio_met in df_recorridos.columns else 0
                            importe_promedio_met_bd = row.get(col_importe_promedio_met, 0) if col_importe_promedio_met in df_recorridos.columns else 0
                            distancia_promedio_met_bd = row.get(col_distancia_promedio_met, 0) if col_distancia_promedio_met in df_recorridos.columns else 0
                            tiros_promedio_met_bd = row.get(col_tiros_promedio_met, 0) if col_tiros_promedio_met in df_recorridos.columns else 0
                            
                            popup_html = f'''
                            <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; max-width: 400px; font-family: 'Segoe UI', Arial, sans-serif;">
                                <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                                    <div style="font-size: 22px; font-weight: bold;">🏠 {row[col_nombre]}</div>
                                    <div style="font-size: 14px; opacity: 0.95; margin-top: 4px;">MET {row[col_codigo]}</div>
                                </div>
                                
                                <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin-bottom: 16px;">
                                    <div style="background: #E8F5E8; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #66BB6A;">
                                        <div style="font-size: 10px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN PROMEDIO</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_promedio_met_bd:,.2f} m³</div>
                                        <div style="font-size: 8px; color: #2E7D32;">{total_clientes_met} clientes</div>
                                    </div>
                                    <div style="background: #E3F2FD; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #42A5F5;">
                                        <div style="font-size: 10px; color: #1976D2; font-weight: bold;">💰 IMPORTE PROMEDIO</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #0D47A1; margin: 4px 0;">${importe_promedio_met_bd:,.0f}</div>
                                    </div>
                                    <div style="background: #FFF3E0; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #FF9800;">
                                        <div style="font-size: 10px; color: #F57C00; font-weight: bold;">📏 DISTANCIA PROMEDIO</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #E65100; margin: 4px 0;">{distancia_promedio_met_bd:,.1f} km</div>
                                    </div>
                                    <div style="background: #F3E5F5; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #9C27B0;">
                                        <div style="font-size: 10px; color: #7B1FA2; font-weight: bold;">🎯 TIROS PROMEDIO</div>
                                        <div style="font-size: 16px; font-weight: bold; color: #4A148C; margin: 4px 0;">{tiros_promedio_met_bd:,.1f}</div>
                                    </div>
                                </div>
                                
                                <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                                    <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 EFICIENCIA</div>
                                    <div style="font-size: 11px; color: #388E3C;">
                                        Vol/km: <b>{(volumen_total_met/max(distancia_total_met,1)):,.2f}</b> | 
                                        Imp/km: <b>${(importe_total_met/max(distancia_total_met,1)):,.0f}</b> | 
                                        Rutas: <b>{len(ruta_recorridos[col_ruta].unique())}</b>
                                    </div>
                                </div>
                                
                                <div style="font-size: 10px; color: #666; text-align: center; border-top: 1px solid #ddd; padding-top: 8px; margin-top: 8px;">
                                    📍 Datos promedio consolidados del MET
                                </div>
                            </div>
                            '''
                            # Usar imagen personalizada para el MET si está disponible
                            if os.path.exists(icon_met_path):
                                icon_met = CustomIcon(
                                    icon_image=icon_met_path,
                                    icon_size=(40, 40),
                                    icon_anchor=(20, 40),
                                    popup_anchor=(0, -40)
                                )
                            else:
                                # Fallback al ícono por defecto si la imagen no existe
                                icon_met = folium.Icon(color=color_ruta, icon='home', prefix='fa')
                            
                            folium.Marker(
                                location=[row[col_latitud], row[col_longitud]],
                                popup=folium.Popup(popup_html, max_width=340),
                                icon=icon_met,
                                **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                            ).add_to(fg_combo)
                            
                            folium.Marker(
                                location=[row[col_latitud], row[col_longitud]],
                                icon=DivIcon(
                                    icon_size=(24,24),
                                    icon_anchor=(12,12),
                                    html=f'<div style="font-size:16px; color:white; background:{color_ruta}; border-radius:50%; width:24px; height:24px; text-align:center; line-height:24px; border:2px solid #fff;">{row[col_secuencia]}</div>'
                                ),
                            ).add_to(fg_combo)
                
                # MOSTRAR METS SIEMPRE (incluso en rutas agrupadas) - SIMPLIFICADO
                for i, row in ruta_recorridos_sorted.iterrows():
                    if row[col_tipo] == 'MET':
                        # Calcular estadísticas agregadas para este MET (usando datos filtrados)
                        clientes_met = clientes_todos[clientes_todos[col_met] == met]
                        if not clientes_met.empty:
                            volumen_total_met = clientes_met[col_volumen].sum()
                            importe_total_met = clientes_met[col_importe].sum()
                            tiros_total_met = clientes_met[col_tiros].sum()
                            # Filtrar valores infinitos y muy altos en distancia
                            distancias_validas = clientes_met[col_distancia_sig].replace([np.inf, -np.inf], np.nan).dropna()
                            distancia_total_met = distancias_validas.sum() if not distancias_validas.empty else 0
                            total_clientes_met = len(clientes_met)
                        else:
                            volumen_total_met = importe_total_met = tiros_total_met = distancia_total_met = total_clientes_met = 0
                        
                        # OBTENER DATOS DIRECTOS DE MET DESDE LA BASE DE DATOS
                        volumen_promedio_met_bd = row.get(col_volumen_promedio_met, 0) if col_volumen_promedio_met in df_recorridos.columns else 0
                        importe_promedio_met_bd = row.get(col_importe_promedio_met, 0) if col_importe_promedio_met in df_recorridos.columns else 0
                        distancia_promedio_met_bd = row.get(col_distancia_promedio_met, 0) if col_distancia_promedio_met in df_recorridos.columns else 0
                        tiros_promedio_met_bd = row.get(col_tiros_promedio_met, 0) if col_tiros_promedio_met in df_recorridos.columns else 0
                        
                        # Verificar si la distancia es válida para mostrar
                        distancia_display = f"{distancia_total_met:,.1f}" if distancia_total_met < 1e6 and not np.isnan(distancia_total_met) and not np.isinf(distancia_total_met) else "N/A"
                        
                        popup_html_met = f'''
                        <div style="background: linear-gradient(135deg, #e8f5e8 0%, #c8e6c9 100%); border-radius: 20px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); padding: 20px; border: 3px solid #4CAF50; min-width: 320px; max-width: 400px; font-family: 'Segoe UI', Arial, sans-serif;">
                            <div style="background: #4CAF50; color: white; margin: -20px -20px 16px -20px; padding: 16px; border-radius: 17px 17px 8px 8px; text-align: center;">
                                <div style="font-size: 22px; font-weight: bold;">🏠 {met}</div>
                            </div>
                            
                            <div style="display: grid; grid-template-columns: 1fr 1fr 1fr 1fr; gap: 10px; margin-bottom: 16px;">
                                <div style="background: #E8F5E8; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #66BB6A;">
                                    <div style="font-size: 10px; color: #2E7D32; font-weight: bold;">📦 VOLUMEN PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #1B5E20; margin: 4px 0;">{volumen_promedio_met_bd:,.2f} m³</div>
                                    <div style="font-size: 8px; color: #2E7D32;">{total_clientes_met} clientes</div>
                                </div>
                                <div style="background: #E3F2FD; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #42A5F5;">
                                    <div style="font-size: 10px; color: #1976D2; font-weight: bold;">💰 IMPORTE PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #0D47A1; margin: 4px 0;">${importe_promedio_met_bd:,.0f}</div>
                                </div>
                                <div style="background: #FFF3E0; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #FF9800;">
                                    <div style="font-size: 10px; color: #F57C00; font-weight: bold;">📏 DISTANCIA PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #E65100; margin: 4px 0;">{distancia_promedio_met_bd:,.1f} km</div>
                                </div>
                                <div style="background: #F3E5F5; padding: 10px; border-radius: 10px; text-align: center; border: 2px solid #9C27B0;">
                                    <div style="font-size: 10px; color: #7B1FA2; font-weight: bold;">🎯 TIROS PROMEDIO</div>
                                    <div style="font-size: 16px; font-weight: bold; color: #4A148C; margin: 4px 0;">{tiros_promedio_met_bd:,.1f}</div>
                                </div>
                            </div>
                            
                            <div style="background: rgba(76, 175, 80, 0.1); padding: 12px; border-radius: 12px; text-align: center; border: 1px solid #4CAF50;">
                                <div style="font-size: 13px; font-weight: bold; color: #2E7D32; margin-bottom: 4px;">📊 EFICIENCIA</div>
                                <div style="font-size: 11px; color: #388E3C;">
                                    Vol/km: <b>{(volumen_total_met/max(distancia_total_met,1) if distancia_total_met > 0 and distancia_total_met < 1e6 else 0):,.2f}</b> | 
                                    Imp/km: <b>${(importe_total_met/max(distancia_total_met,1) if distancia_total_met > 0 and distancia_total_met < 1e6 else 0):,.0f}</b> | 
                                    Rutas: <b>{len(clientes_todos[clientes_todos[col_met]==met][col_ruta].unique())}</b>
                                </div>
                            </div>
                            
                            <div style="font-size: 10px; color: #666; text-align: center; border-top: 1px solid #ddd; padding-top: 8px; margin-top: 8px;">
                                📍 Datos promedio consolidados del MET
                            </div>
                        </div>
                        '''
                        # Usar imagen personalizada para el MET si está disponible
                        if os.path.exists(icon_met_path):
                            icon_met = CustomIcon(
                                icon_image=icon_met_path,
                                icon_size=(40, 40),
                                icon_anchor=(20, 40),
                                popup_anchor=(0, -40)
                            )
                        else:
                            # Fallback al ícono por defecto si la imagen no existe
                            icon_met = folium.Icon(color=color_ruta, icon='home', prefix='fa')
                        
                        folium.Marker(
                            location=[row[col_latitud], row[col_longitud]],
                            popup=folium.Popup(popup_html_met, max_width=420),
                            icon=icon_met,
                            **{'data-met': met_id, 'data-ruta': ruta_id, 'data-combo': combo_id}
                        ).add_to(fg_combo)
                
                # Agregar la capa al mapa
                fg_combo.add_to(mapa)
                
                # Calcular estadísticas para el resumen
                clientes_ruta = ruta_recorridos[ruta_recorridos[col_tipo] == 'Cliente']
                total_clientes_ruta = clientes_ruta.shape[0]
                
                # Buscar columna de distancia total de ruta
                columnas_distancia_total = [col for col in df_recorridos.columns if 'distancia' in col.lower() and 'total' in col.lower() and 'ruta' in col.lower()]
                if columnas_distancia_total:
                    distancia_total_ruta = ruta_recorridos[columnas_distancia_total[0]].dropna().unique()
                    distancia_total_ruta = distancia_total_ruta[0] if len(distancia_total_ruta) > 0 else 0
                else:
                    distancia_total_ruta = 0
                distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                
                volumen_total_ruta = clientes_ruta[col_volumen].sum() if col_volumen in df_recorridos.columns else 0
                importe_total_ruta = clientes_ruta[col_importe].sum() if col_importe in df_recorridos.columns else 0
                volumen_promedio_ruta = volumen_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                importe_promedio_ruta = importe_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
                
                resumen_combos[combo_id] = {
                    'met': met,
                    'ruta': ruta,
                    'clientes': total_clientes_ruta,
                    'distancia_total_km': distancia_total_ruta,
                    'distancia_promedio_cliente_km': distancia_promedio_ruta,
                    'volumen_total': volumen_total_ruta,
                    'volumen_promedio': volumen_promedio_ruta,
                    'importe_total': importe_total_ruta,
                    'importe_promedio': importe_promedio_ruta,
                    'color': color_ruta
                }

        # Agregar control de capas
        folium.LayerControl(collapsed=False, autoZIndex=True).add_to(mapa)

        # CSS para scroll en el control de capas y botones de selección (ACTUALIZADO CON FILTROS)
        scroll_layers_css = '''
        <style>
        /* OCULTAR CHECKBOXES NATIVOS DE FOLIUM */
        .leaflet-control-layers-overlays {
            display: none !important;
        }
        
        .leaflet-control-layers-list { 
            max-height: 400px; 
            overflow-y: auto; 
            overflow-x: hidden; 
            width: 100%; 
            min-width: 200px; 
        }
        .layer-control-buttons {
            padding: 8px;
            border-bottom: 1px solid #ddd;
            background: #f8f8f8;
            display: flex;
            gap: 5px;
        }
        .layer-btn {
            padding: 6px 10px;
            font-size: 11px;
            border: 1px solid #ccc;
            background: white;
            cursor: pointer;
            border-radius: 4px;
            flex: 1;
            text-align: center;
            font-weight: 600;
            transition: all 0.2s;
        }
        .layer-btn:hover {
            background: #e6e6e6;
            transform: translateY(-1px);
        }
        .layer-btn.select-all {
            background: #4CAF50;
            color: white;
            border-color: #45a049;
        }
        .layer-btn.select-all:hover {
            background: #45a049;
        }
        .layer-btn.deselect-all {
            background: #f44336;
            color: white;
            border-color: #da190b;
        }
        .layer-btn.deselect-all:hover {
            background: #da190b;
        }
        .met-buttons-row {
            padding: 6px 8px;
            border-bottom: 1px solid #ddd;
            background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
            display: flex;
            gap: 4px;
            flex-wrap: wrap;
        }
        .met-btn {
            padding: 5px 10px;
            font-size: 10px;
            border: 2px solid #2196F3;
            background: white;
            color: #1976D2;
            cursor: pointer;
            border-radius: 6px;
            flex: 1;
            text-align: center;
            min-width: 60px;
            font-weight: 600;
            transition: all 0.2s;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .met-btn:hover {
            background: #2196F3;
            color: white;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
        }
        .ruta-buttons-row {
            padding: 6px 8px;
            border-bottom: 1px solid #ddd;
            background: linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%);
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(45px, 1fr));
            gap: 4px;
            max-width: 100%;
        }
        .ruta-btn {
            padding: 5px 8px;
            font-size: 10px;
            border: 2px solid #9C27B0;
            background: white;
            color: #7B1FA2;
            cursor: pointer;
            border-radius: 6px;
            text-align: center;
            min-width: 45px;
            font-weight: 700;
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
            transition: all 0.2s;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .ruta-btn:hover {
            background: #9C27B0;
            color: white;
            transform: translateY(-2px);
            box-shadow: 0 4px 8px rgba(0,0,0,0.2);
        }
        </style>
        '''
        mapa.get_root().html.add_child(folium.Element(scroll_layers_css))
        
        # JavaScript para agregar botones de selección
        layer_control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            setTimeout(function() {
                // Buscar el control de capas
                const layerControl = document.querySelector('.leaflet-control-layers');
                if (layerControl) {
                    const layersList = layerControl.querySelector('.leaflet-control-layers-list');
                    const overlaysDiv = layerControl.querySelector('.leaflet-control-layers-overlays');
                    
                    if (overlaysDiv && layersList) {
                        // Crear div para botones
                        const buttonsDiv = document.createElement('div');
                        buttonsDiv.className = 'layer-control-buttons';
                        
                        // Botón Seleccionar Todo
                        const selectAllBtn = document.createElement('button');
                        selectAllBtn.textContent = '✓ Todo';
                        selectAllBtn.className = 'layer-btn select-all';
                        selectAllBtn.title = 'Seleccionar todas las capas';
                        
                        // Botón Deseleccionar Todo
                        const deselectAllBtn = document.createElement('button');
                        deselectAllBtn.textContent = '✗ Nada';
                        deselectAllBtn.className = 'layer-btn deselect-all';
                        deselectAllBtn.title = 'Deseleccionar todas las capas';
                        
                        // Funcionalidad de los botones
                        selectAllBtn.onclick = function(e) {
                            e.preventDefault();
                            e.stopPropagation();
                            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                            checkboxes.forEach(function(checkbox) {
                                if (!checkbox.checked) {
                                    checkbox.click();
                                }
                            });
                        };
                        
                        deselectAllBtn.onclick = function(e) {
                            e.preventDefault();
                            e.stopPropagation();
                            const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                            checkboxes.forEach(function(checkbox) {
                                if (checkbox.checked) {
                                    checkbox.click();
                                }
                            });
                        };
                        
                        // Agregar botones al div
                        buttonsDiv.appendChild(selectAllBtn);
                        buttonsDiv.appendChild(deselectAllBtn);
                        
                        // Insertar botones antes de las capas
                        layersList.insertBefore(buttonsDiv, overlaysDiv);
                        
                        // NUEVA FUNCIONALIDAD: Botones por MET
                        const metButtonsDiv = document.createElement('div');
                        metButtonsDiv.className = 'met-buttons-row';
                        
                        // Obtener METs únicos de las capas
                        const checkboxes = overlaysDiv.querySelectorAll('input[type="checkbox"]');
                        const mets = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            const metMatch = label.match(/^MET\\s+([^\\s-]+)/);
                            if (metMatch) {
                                mets.add(metMatch[1]);
                            }
                        });
                        
                        // Crear botón para cada MET
                        mets.forEach(function(met) {
                            const metBtn = document.createElement('button');
                            metBtn.textContent = met;
                            metBtn.className = 'met-btn';
                            metBtn.title = 'Seleccionar solo rutas de MET ' + met;
                            
                            metBtn.onclick = function(e) {
                                e.preventDefault();
                                e.stopPropagation();
                                
                                // Primero deseleccionar todo
                                checkboxes.forEach(function(checkbox) {
                                    if (checkbox.checked) {
                                        checkbox.click();
                                    }
                                });
                                
                                // Desmarcar checkboxes de áreas y clientes
                                const toggleAreas = document.getElementById('toggle-areas');
                                const toggleClientes = document.getElementById('toggle-clientes');
                                if (toggleAreas && toggleAreas.checked) toggleAreas.checked = false;
                                if (toggleClientes && toggleClientes.checked) toggleClientes.checked = false;
                                
                                // Luego seleccionar solo las rutas del MET específico
                                setTimeout(function() {
                                    checkboxes.forEach(function(checkbox) {
                                        const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                        const metPattern = new RegExp('^MET\\\\s+' + met.replace(/[.*+?^${}()|[\\]\\\\]/g, '\\\\$&') + '\\\\s*-');
                                        if (metPattern.test(label) && !checkbox.checked) {
                                            checkbox.click();
                                        }
                                    });
                                    
                                    // Aplicar el estado de los checkboxes
                                    if (toggleAreas) {
                                        const event = new Event('change');
                                        toggleAreas.dispatchEvent(event);
                                    }
                                    if (toggleClientes) {
                                        const event = new Event('change');
                                        toggleClientes.dispatchEvent(event);
                                    }
                                }, 100);
                            };
                            
                            metButtonsDiv.appendChild(metBtn);
                        });
                        
                        // Insertar botones de MET después de los botones principales
                        layersList.insertBefore(metButtonsDiv, overlaysDiv);
                        
                        // NUEVA FUNCIONALIDAD: Botones por Rutas Completas
                        const rutaButtonsDiv = document.createElement('div');
                        rutaButtonsDiv.className = 'ruta-buttons-row';
                        
                        // Obtener rutas únicas de las capas
                        const rutas = new Set();
                        checkboxes.forEach(function(checkbox) {
                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                            const rutaMatch = label.match(/MET\\s+[^-]+\\s*-\\s*Ruta\\s+(\\S+)/);
                            if (rutaMatch) {
                                rutas.add(rutaMatch[1]);
                            }
                        });
                        
                        // Ordenar rutas numéricamente
                        const rutasArray = Array.from(rutas).sort((a, b) => {
                            const numA = parseInt(a);
                            const numB = parseInt(b);
                            if (!isNaN(numA) && !isNaN(numB)) {
                                return numA - numB;
                            }
                            return a.localeCompare(b);
                        });
                        
                        // Crear botón para cada ruta
                        rutasArray.forEach(function(ruta) {
                            const rutaBtn = document.createElement('button');
                            rutaBtn.textContent = 'R' + ruta;
                            rutaBtn.className = 'ruta-btn';
                            rutaBtn.title = 'Seleccionar Ruta ' + ruta + ' de todos los METs';
                            
                            rutaBtn.onclick = function(e) {
                                e.preventDefault();
                                e.stopPropagation();
                                
                                // Desmarcar checkboxes de áreas y clientes
                                const toggleAreas = document.getElementById('toggle-areas');
                                const toggleClientes = document.getElementById('toggle-clientes');
                                if (toggleAreas && toggleAreas.checked) toggleAreas.checked = false;
                                if (toggleClientes && toggleClientes.checked) toggleClientes.checked = false;
                                
                                // Primero deseleccionar todos los checkboxes
                                setTimeout(() => {
                                    checkboxes.forEach(function(checkbox) {
                                        if (checkbox.checked) {
                                            checkbox.click();
                                        }
                                    });
                                    
                                    // Luego seleccionar solo las rutas que coincidan
                                    setTimeout(() => {
                                        checkboxes.forEach(function(checkbox) {
                                            const label = checkbox.parentElement.querySelector('span:last-child').textContent.trim();
                                            const rutaPattern = new RegExp('MET\\\\s+[^-]+\\\\s*-\\\\s*Ruta\\\\s+' + ruta.replace(/[.*+?^${}()|[\\]\\\\]/g, '\\\\$&') + '\\\\s*$');
                                            if (rutaPattern.test(label) && !checkbox.checked) {
                                                checkbox.click();
                                            }
                                        });
                                        
                                        // Aplicar el estado de los checkboxes
                                        if (toggleAreas) {
                                            const event = new Event('change');
                                            toggleAreas.dispatchEvent(event);
                                        }
                                        if (toggleClientes) {
                                            const event = new Event('change');
                                            toggleClientes.dispatchEvent(event);
                                        }
                                    }, 150);
                                }, 100);
                            };
                            
                            rutaButtonsDiv.appendChild(rutaBtn);
                        });
                        
                        // Insertar botones de Rutas después de los botones de MET
                        layersList.insertBefore(rutaButtonsDiv, overlaysDiv);
                    }
                }
            }, 2000); // Esperar 2 segundos para que el mapa se cargue completamente
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(layer_control_js))

        # Control personalizado adaptado para rutas agrupadas
        if tipo_visualizacion.value == 'rutas_agrupadas':
            control_personalizado = '''
            <div id="controls-personalizados" style="position: fixed; top: 100px; left: 10px; z-index: 1000; background: white; padding: 12px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.3); font-family: 'Segoe UI', Arial, sans-serif; min-width: 220px;">
                <h4 style="margin: 0 0 10px 0; color: #333; font-size: 14px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">🎛️ Controles Vista Rutas</h4>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-areas" style="margin-right: 8px;">
                        <span style="color: #333;">📍 Áreas de Cobertura</span>
                    </label>
                </div>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-esferas" checked style="margin-right: 8px;">
                        <span style="color: #333;">🛣️ Esferas de Rutas</span>
                    </label>
                </div>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-clientes" style="margin-right: 8px;">
                        <span style="color: #333;">👥 Área Solo Clientes</span>
                    </label>
                </div>
                
                <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid #eee; font-size: 10px; color: #666;">
                    <div>💡 <b>Vista Rutas Agrupadas:</b></div>
                    <div>🔴🟠🟡🟢 Colores por eficiencia relativa</div>
                    <div>📦 Tamaño por volumen total</div>
                </div>
            </div>
            '''
        else:
            # Control original para otras vistas
            control_personalizado = '''
            <div id="controls-personalizados" style="position: fixed; top: 100px; left: 10px; z-index: 1000; background: white; padding: 10px; border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,0.3); font-family: 'Segoe UI', Arial, sans-serif; min-width: 200px;">
                <h4 style="margin: 0 0 10px 0; color: #333; font-size: 14px; border-bottom: 1px solid #ddd; padding-bottom: 5px;">🎛️ Controles del Mapa</h4>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-lineas" checked style="margin-right: 8px;">
                        <span style="color: #333;">🛣️ Líneas de Rutas</span>
                    </label>
                </div>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-areas" style="margin-right: 8px;">
                        <span style="color: #333;">📍 Áreas de Cobertura</span>
                    </label>
                </div>
                
                <div style="margin-bottom: 8px;">
                    <label style="display: flex; align-items: center; cursor: pointer; font-size: 12px;">
                        <input type="checkbox" id="toggle-clientes" checked style="margin-right: 8px;">
                        <span style="color: #333;">👥 Área Solo Clientes</span>
                    </label>
                </div>
                
                <div style="margin-top: 10px; padding-top: 8px; border-top: 1px solid #eee; font-size: 10px; color: #666;">
                    <div>💡 <b>Tip:</b> Usa estos controles para</div>
                    <div>personalizar la visualización</div>
                </div>
            </div>
            '''
        mapa.get_root().html.add_child(folium.Element(control_personalizado))

        # JavaScript para controlar la visibilidad
        control_js = '''
        <script>
        document.addEventListener('DOMContentLoaded', function() {
            // Control de elementos del mapa
            const toggleLineas = document.getElementById('toggle-lineas');
            const toggleAreas = document.getElementById('toggle-areas');
            const toggleEsferas = document.getElementById('toggle-esferas');
            const toggleMets = document.getElementById('toggle-mets');
            const toggleClientes = document.getElementById('toggle-clientes');
            
            // Función para mostrar/ocultar líneas de rutas
            function toggleRutaLines() {
                if (!toggleLineas) return;
                const rutaLines = document.querySelectorAll('.ruta-line');
                const isVisible = toggleLineas.checked;
                rutaLines.forEach(line => {
                    if (line.style) {
                        line.style.display = isVisible ? 'block' : 'none';
                    }
                    if (line.tagName === 'path') {
                        line.style.display = isVisible ? 'block' : 'none';
                    }
                });
                
                const polylines = document.querySelectorAll('path[stroke]');
                polylines.forEach(polyline => {
                    if (polyline.getAttribute('stroke-width') === '5') {
                        polyline.style.display = isVisible ? 'block' : 'none';
                    }
                });
            }
            
            // Función para mostrar/ocultar áreas de cobertura
            function toggleAreaPolygons() {
                if (!toggleAreas) return;
                const areaPolygons = document.querySelectorAll('.area-ruta');
                const isVisible = toggleAreas.checked;
                areaPolygons.forEach(polygon => {
                    polygon.style.display = isVisible ? 'block' : 'none';
                });
                
                // También controlar polígonos SVG (tanto para vista normal como rutas agrupadas)
                const svgPolygons = document.querySelectorAll('path[fill-opacity]');
                svgPolygons.forEach(polygon => {
                    const fillOpacity = polygon.getAttribute('fill-opacity');
                    if (fillOpacity === '0.15' || fillOpacity === '0.25') {
                        polygon.style.display = isVisible ? 'block' : 'none';
                    }
                });
            }
            
            // Función para mostrar/ocultar áreas de solo clientes
            function toggleClientePolygons() {
                if (!toggleClientes) return;
                const clientePolygons = document.querySelectorAll('.area-clientes');
                const isVisible = toggleClientes.checked;
                clientePolygons.forEach(polygon => {
                    polygon.style.display = isVisible ? 'block' : 'none';
                });
                
                // También controlar polígonos SVG específicos de clientes
                const svgPolygons = document.querySelectorAll('path[fill-opacity]');
                svgPolygons.forEach(polygon => {
                    const fillOpacity = polygon.getAttribute('fill-opacity');
                    if (fillOpacity === '0.1') {  // Las áreas de clientes tienen opacity 0.1
                        polygon.style.display = isVisible ? 'block' : 'none';
                    }
                });
            }
            
            // Función para mostrar/ocultar esferas de rutas
            function toggleRutaEsferas() {
                if (!toggleEsferas) return;
                const rutaEsferas = document.querySelectorAll('.ruta-esfera');
                const isVisible = toggleEsferas.checked;
                rutaEsferas.forEach(esfera => {
                    esfera.style.display = isVisible ? 'block' : 'none';
                });
                
                // También controlar círculos SVG de rutas
                const circleMarkers = document.querySelectorAll('circle[class*="ruta"]');
                circleMarkers.forEach(circle => {
                    circle.style.display = isVisible ? 'block' : 'none';
                });
            }
            
            // Función para mostrar/ocultar METs
            function toggleMetMarkers() {
                if (!toggleMets) return;
                const metMarkers = document.querySelectorAll('img[src*="95_24.png"]');
                const isVisible = toggleMets.checked;
                metMarkers.forEach(marker => {
                    marker.parentElement.style.display = isVisible ? 'block' : 'none';
                });
                
                // También controlar marcadores con icono de casa
                const homeIcons = document.querySelectorAll('.fa-home');
                homeIcons.forEach(icon => {
                    icon.parentElement.parentElement.style.display = isVisible ? 'block' : 'none';
                });
            }
            
            // Event listeners condicionales
            if (toggleLineas) toggleLineas.addEventListener('change', toggleRutaLines);
            if (toggleAreas) toggleAreas.addEventListener('change', toggleAreaPolygons);
            if (toggleEsferas) toggleEsferas.addEventListener('change', toggleRutaEsferas);
            if (toggleMets) toggleMets.addEventListener('change', toggleMetMarkers);
            if (toggleClientes) toggleClientes.addEventListener('change', toggleClientePolygons);
            
            // Estado inicial
            setTimeout(() => {
                toggleAreaPolygons();
                if (toggleLineas) toggleRutaLines();
                if (toggleEsferas) toggleRutaEsferas();
                if (toggleMets) toggleMetMarkers();
                if (toggleClientes) toggleClientePolygons();
            }, 1000);
            
            // Aplicar estilos CSS adicionales
            const style = document.createElement('style');
            style.textContent = `
                .ruta-line {
                    transition: opacity 0.3s ease;
                }
                .area-ruta {
                    transition: opacity 0.3s ease;
                }
                .ruta-esfera {
                    transition: opacity 0.3s ease;
                }
                .leaflet-div-icon {
                    background: transparent !important;
                    border: none !important;
                }
                path[stroke-width="5"] {
                    transition: opacity 0.3s ease;
                }
                path[fill-opacity="0.15"], path[fill-opacity="0.25"] {
                    transition: opacity 0.3s ease;
                }
            `;
            document.head.appendChild(style);
        });
        </script>
        '''
        mapa.get_root().html.add_child(folium.Element(control_js))
        
        # Título del mapa MEJORADO
        desc_visual = {
            'rutas_agrupadas': '🛣️ Vista por Rutas: Una esfera por ruta | 📦 Tamaño = Volumen Total | 🎨 Color = Score Rentabilidad 1-100 (Volumen/Distancia)',
            'marcadores_smart': '🎯 Marcadores Inteligentes: 📦 Tamaño = Volumen | 🎨 Color = Importe | ⭐ Categorías VIP',
            'marcadores': '📍 Marcadores: 📦 Tamaño = Volumen | 🎨 Color = Importe (🟡→🟠→🔴)',
            'solo_heatmap': '🔥 Mapas de Calor: Concentración de Volumen e Importe',
            'completo': '🎨 Vista Completa: 🛣️ Rutas + 🔥 Mapas de Calor'
        }
        
        # Calcular número de clientes únicos (sin duplicados)
        clientes_unicos = clientes_todos[col_codigo].nunique() if not clientes_todos.empty and col_codigo in clientes_todos.columns else len(clientes_todos)
        
        titulo_html = f'''
        <div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 12px 24px; border-radius: 15px; box-shadow: 0 8px 32px rgba(0,0,0,0.3); border: 2px solid #fff;">
            <h2 style="margin: 0; text-align: center; font-size:16px; text-shadow: 1px 1px 2px rgba(0,0,0,0.5);">
                🗺️ ANÁLISIS COMERCIAL - VOLUMEN E IMPORTE
            </h2>
            <p style="margin: 6px 0 0 0; text-align: center; font-size: 11px; opacity: 0.95;">
                📍 METs: <b>{len(mets_seleccionados)}</b> | 🛣️ Rutas: <b>{len(rutas_seleccionadas)}</b> | 👥 Clientes: <b>{clientes_unicos}</b><br>
                {desc_visual.get(tipo_visualizacion.value, '')}
            </p>
        </div>
        '''
        mapa.get_root().html.add_child(folium.Element(titulo_html))

        print(f"✅ Mapa generado exitosamente con {len(featuregroups_combo)} capas")
        
        # Guardar el mapa
        tipo_viz = tipo_visualizacion.value.replace('_', '-')
        nombre_mapa = os.path.join(output_dir, f'mapa_analisis_comercial_{tipo_viz}_{fecha_hora}.html')
        try:
            mapa.save(nombre_mapa)
            print(f'🗺️ Mapa de análisis comercial exportado a: {nombre_mapa}')
            print(f"🎯 Visualización utilizada: {desc_visual.get(tipo_visualizacion.value, tipo_visualizacion.value)}")
        except Exception as e:
            print(f'❌ Error al guardar el mapa: {e}')

# Conectar el botón con la función
generar_btn.on_click(generar_mapa)

def generar_analisis_excel(b):
    with output_excel:
        clear_output()
        
        # Verificar que los datos estén disponibles
        if df_recorridos is None or df_recorridos.empty:
            return
            
        mets_seleccionados = list(met_selector.value)
        rutas_seleccionadas = list(ruta_selector.value)
        if not mets_seleccionados:
            print('Selecciona al menos un MET para generar el análisis.')
            return
        if not rutas_seleccionadas:
            print('Selecciona al menos una ruta para generar el análisis.')
            return
        
        # Filtrar datos según selección
        datos_filtrados = df_recorridos[
            (df_recorridos[col_met].isin(mets_seleccionados)) & 
            (df_recorridos[col_ruta].isin(rutas_seleccionadas))
        ]
        
        clientes_filtrados = datos_filtrados[datos_filtrados[col_tipo] == 'Cliente'].copy()
        
        if clientes_filtrados.empty:
            print("❌ No se encontraron clientes en la selección actual")
            return
        
        # Calcular percentiles y rankings
        clientes_filtrados['percentil_volumen'] = clientes_filtrados[col_volumen].rank(pct=True) * 100
        clientes_filtrados['percentil_importe'] = clientes_filtrados[col_importe].rank(pct=True) * 100
        clientes_filtrados['ranking_volumen'] = clientes_filtrados[col_volumen].rank(ascending=False, method='dense').astype(int)
        clientes_filtrados['ranking_importe'] = clientes_filtrados[col_importe].rank(ascending=False, method='dense').astype(int)
        
        # Calcular categorías de cliente
        def calcular_categoria(row):
            if row['percentil_volumen'] >= 80 and row['percentil_importe'] >= 80:
                return "VIP"
            elif row['percentil_volumen'] >= 70 or row['percentil_importe'] >= 70:
                return "Premium"
            elif row['percentil_volumen'] >= 50 or row['percentil_importe'] >= 50:
                return "Estándar"
            else:
                return "Básico"
        
        clientes_filtrados['categoria'] = clientes_filtrados.apply(calcular_categoria, axis=1)
        
        # Crear análisis por MET
        resumen_mets = []
        for met in mets_seleccionados:
            clientes_met = clientes_filtrados[clientes_filtrados[col_met] == met]
            if not clientes_met.empty:
                # Obtener datos precalculados de MET desde la base de datos (igual que en el mapa)
                met_data = datos_filtrados[datos_filtrados[col_met] == met].iloc[0]
                
                # Usar los mismos datos que el mapa para distancia promedio
                distancia_promedio_met_bd = met_data.get(col_distancia_promedio_met, 0) if col_distancia_promedio_met in df_recorridos.columns else 0
                
                # USAR EXACTAMENTE EL MISMO CÁLCULO QUE EN EL MAPA
                # Filtrar valores infinitos y muy altos en distancia (igual que líneas 1164-1165 del mapa)
                distancias_validas = clientes_met[col_distancia_sig].replace([np.inf, -np.inf], np.nan).dropna()
                distancia_total_met = distancias_validas.sum() if not distancias_validas.empty else 0
                
                resumen = {
                    'MET': met,
                    'Total_Clientes': len(clientes_met),
                    'Volumen_Total': clientes_met[col_volumen].sum(),
                    'Volumen_Promedio': clientes_met[col_volumen].mean(),
                    'Importe_Total': clientes_met[col_importe].sum(),
                    'Importe_Promedio': clientes_met[col_importe].mean(),
                    'Tiros_Total': clientes_met[col_tiros].sum(),
                    'Tiros_Promedio': clientes_met[col_tiros].mean(),
                    'Distancia_Total': distancia_total_met,
                    'Distancia_Promedio': distancia_promedio_met_bd if distancia_promedio_met_bd > 0 else clientes_met[col_distancia_sig].mean(),
                    'Eficiencia_Volumen_Distancia': clientes_met[col_volumen].sum() / max(distancia_total_met, 1),
                    'Eficiencia_Importe_Distancia': clientes_met[col_importe].sum() / max(distancia_total_met, 1),
                    'Score_Rentabilidad': clientes_met[col_volumen].sum() / max(distancia_total_met, 1)
                }
                resumen_mets.append(resumen)
        
        df_resumen_mets = pd.DataFrame(resumen_mets)
        
        # Ranking de METs por diferentes criterios
        df_resumen_mets['Ranking_Volumen_Total'] = df_resumen_mets['Volumen_Total'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Importe_Total'] = df_resumen_mets['Importe_Total'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Eficiencia_Vol'] = df_resumen_mets['Eficiencia_Volumen_Distancia'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Eficiencia_Imp'] = df_resumen_mets['Eficiencia_Importe_Distancia'].rank(ascending=False, method='dense').astype(int)
        df_resumen_mets['Ranking_Rentabilidad'] = df_resumen_mets['Score_Rentabilidad'].rank(ascending=False, method='dense').astype(int)
        
        # Crear archivo Excel
        nombre_excel = os.path.join(output_dir, f'analisis_comercial_mets_{fecha_hora}.xlsx')
        wb = Workbook()
        
        # Configurar estilos
        header_font = Font(bold=True, color="FFFFFF")
        header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
        
        # 1. Hoja: Resumen Ejecutivo de METs
        ws_resumen = wb.active
        ws_resumen.title = "Resumen METs"
        
        # Escribir datos del resumen
        for r in dataframe_to_rows(df_resumen_mets, index=False, header=True):
            ws_resumen.append(r)
        
        # Aplicar formato a encabezados
        for cell in ws_resumen[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center")
        
        # Ajustar ancho de columnas
        for column in ws_resumen.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            ws_resumen.column_dimensions[column_letter].width = min(max_length + 2, 20)
        
        # 2. Hoja: Análisis Detallado por Cliente
        ws_detalle = wb.create_sheet("Detalle Clientes")
        detalle_clientes = clientes_filtrados[
            [col_met, col_ruta, col_codigo, col_nombre, col_volumen, col_importe, col_tiros,
             col_distancia_sig, 'percentil_volumen', 'percentil_importe', 
             'ranking_volumen', 'ranking_importe']
        ].copy()
        
        for r in dataframe_to_rows(detalle_clientes, index=False, header=True):
            ws_detalle.append(r)
        
        # Aplicar formato a encabezados
        for cell in ws_detalle[1]:
            cell.font = header_font
            cell.fill = header_fill
        
        # Ajustar ancho de columnas para detalle clientes
        for column in ws_detalle.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            ws_detalle.column_dimensions[column_letter].width = min(max_length + 2, 20)
        
        # 3. Hoja: Resumen por Rutas
        ws_rutas = wb.create_sheet("Resumen RUTAs")
        
        # Crear análisis por RUTA
        resumen_rutas = []
        for met in mets_seleccionados:
            rutas_del_met = datos_filtrados[datos_filtrados[col_met] == met][col_ruta].unique()
            for ruta in rutas_del_met:
                # Filtrar clientes de esta ruta específica
                clientes_ruta = clientes_filtrados[
                    (clientes_filtrados[col_met] == met) & 
                    (clientes_filtrados[col_ruta] == ruta)
                ]
                
                if not clientes_ruta.empty:
                    # Obtener datos precalculados de RUTA desde la base de datos
                    ruta_data = datos_filtrados[
                        (datos_filtrados[col_met] == met) & 
                        (datos_filtrados[col_ruta] == ruta)
                    ].iloc[0]
                    
                    # Usar datos precalculados cuando estén disponibles
                    volumen_promedio_ruta_bd = ruta_data.get(col_volumen_promedio_ruta, 0) if col_volumen_promedio_ruta in df_recorridos.columns else 0
                    importe_promedio_ruta_bd = ruta_data.get(col_importe_promedio_ruta, 0) if col_importe_promedio_ruta in df_recorridos.columns else 0
                    distancia_promedio_ruta_bd = ruta_data.get(col_distancia_promedio_ruta, 0) if col_distancia_promedio_ruta in df_recorridos.columns else 0
                    tiros_promedio_ruta_bd = ruta_data.get(col_tiros_promedio_ruta, 0) if col_tiros_promedio_ruta in df_recorridos.columns else 0
                    
                    # Calcular distancia total usando el mismo método que el mapa
                    distancias_validas_ruta = clientes_ruta[col_distancia_sig].replace([np.inf, -np.inf], np.nan).dropna()
                    distancia_total_ruta = distancias_validas_ruta.sum() if not distancias_validas_ruta.empty else 0
                    
                    resumen_ruta = {
                        'MET': met,
                        'RUTA': ruta,
                        'Total_Clientes': len(clientes_ruta),
                        'Volumen_Total': clientes_ruta[col_volumen].sum(),
                        'Volumen_Promedio': volumen_promedio_ruta_bd if volumen_promedio_ruta_bd > 0 else clientes_ruta[col_volumen].mean(),
                        'Importe_Total': clientes_ruta[col_importe].sum(),
                        'Importe_Promedio': importe_promedio_ruta_bd if importe_promedio_ruta_bd > 0 else clientes_ruta[col_importe].mean(),
                        'Tiros_Total': clientes_ruta[col_tiros].sum(),
                        'Tiros_Promedio': tiros_promedio_ruta_bd if tiros_promedio_ruta_bd > 0 else clientes_ruta[col_tiros].mean(),
                        'Distancia_Total': distancia_total_ruta,
                        'Distancia_Promedio': distancia_promedio_ruta_bd if distancia_promedio_ruta_bd > 0 else clientes_ruta[col_distancia_sig].mean(),
                        'Eficiencia_Volumen_Distancia': clientes_ruta[col_volumen].sum() / max(distancia_total_ruta, 1),
                        'Eficiencia_Importe_Distancia': clientes_ruta[col_importe].sum() / max(distancia_total_ruta, 1),
                        'Score_Rentabilidad': clientes_ruta[col_volumen].sum() / max(distancia_total_ruta, 1)
                    }
                    resumen_rutas.append(resumen_ruta)
        
        df_resumen_rutas = pd.DataFrame(resumen_rutas)
        
        # Añadir rankings para rutas
        if not df_resumen_rutas.empty:
            df_resumen_rutas['Ranking_Volumen_Total'] = df_resumen_rutas['Volumen_Total'].rank(ascending=False, method='dense').astype(int)
            df_resumen_rutas['Ranking_Importe_Total'] = df_resumen_rutas['Importe_Total'].rank(ascending=False, method='dense').astype(int)
            df_resumen_rutas['Ranking_Eficiencia_Vol'] = df_resumen_rutas['Eficiencia_Volumen_Distancia'].rank(ascending=False, method='dense').astype(int)
            df_resumen_rutas['Ranking_Eficiencia_Imp'] = df_resumen_rutas['Eficiencia_Importe_Distancia'].rank(ascending=False, method='dense').astype(int)
            df_resumen_rutas['Ranking_Rentabilidad'] = df_resumen_rutas['Score_Rentabilidad'].rank(ascending=False, method='dense').astype(int)
        
        # Escribir datos del resumen de rutas
        for r in dataframe_to_rows(df_resumen_rutas, index=False, header=True):
            ws_rutas.append(r)
        
        # Aplicar formato a encabezados
        for cell in ws_rutas[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center")
        
        # Ajustar ancho de columnas para resumen rutas
        for column in ws_rutas.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            ws_rutas.column_dimensions[column_letter].width = min(max_length + 2, 20)
        
        # Guardar archivo
        try:
            wb.save(nombre_excel)
            print(f"✅ ¡Archivo Excel generado exitosamente!")
            print(f"📁 Ubicación: {nombre_excel}")
            print(f"📊 Incluye:")
            print(f"   • Resumen METs: Análisis consolidado por MET ({len(df_resumen_mets)} METs)")
            print(f"   • Resumen RUTAs: Análisis detallado por ruta ({len(df_resumen_rutas)} rutas)")
            print(f"   • Detalle Clientes: Información completa de {len(detalle_clientes)} clientes")
            
        except Exception as e:
            print(f"❌ Error al guardar archivo Excel: {str(e)}")
            print("Verifica que el archivo no esté abierto en otra aplicación.")

# Conectar el nuevo botón
analisis_excel_btn.on_click(generar_analisis_excel)

# Interfaz de usuario MEJORADA
display(widgets.VBox([
    widgets.HTML('<h2 style="color: #2E7D32; text-align: center; margin-bottom: 20px;">🗺️ Análisis Comercial por Rutas Agrupadas</h2>'),
    
    widgets.HTML('<h3 style="color: #1976D2;">📍 Selección de Ubicaciones</h3>'),
    widgets.HBox([met_selector, ruta_selector]),
    widgets.HBox([todas_rutas_checkbox]),
    
    widgets.HTML('<h3 style="color: #7B1FA2;">🎨 Opciones de Visualización</h3>'),
    tipo_visualizacion,
    widgets.HBox([escala_marcadores]),
    
    widgets.HTML('<h3 style="color: #F57C00;">⚙️ Configuración</h3>'),
    widgets.HBox([clientes_a_procesar, todos_clientes_checkbox]),
    
    widgets.HTML('''
    <div style="background: #E8F5E8; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #4CAF50;">
        <h4 style="margin: 0 0 8px 0; color: #2E7D32;">📋 Guía de Visualización (SOLO DATOS PROMEDIO):</h4>
        <ul style="margin: 0; padding-left: 20px; font-size: 12px; color: #388E3C;">
            <li><b>🛣️ Vista por Rutas:</b> Una esfera por ruta con datos promedio consolidados</li>
            <li><b>📦 Volumen:</b> Promedio diario por ruta (sin totales)</li>
            <li><b>💰 Importe:</b> Promedio diario por ruta (sin totales)</li>
            <li><b>📏 Distancia:</b> Promedio diario por ruta (sin totales)</li>
            <li><b>🎯 Tiros:</b> Promedio diario por ruta (sin totales)</li>
            <li><b>⚙️ Tamaño esfera:</b> Proporcional al Volumen Promedio (20-80 píxeles)</li>
            <li><b>⚡ Score:</b> Volumen Promedio / Distancia Promedio (normalizado 1-100)</li>
            <li><b>🎨 Color de esfera:</b> Basado en Score de Eficiencia (1-100)</li>
            <li><b>🟢 Muy Eficiente:</b> Score ≥ 85 (Mayor volumen por km)</li>
            <li><b>🟡 Eficiente:</b> Score 70-85</li>
            <li><b>🟠 Moderado:</b> Score 50-70</li>
            <li><b>🔴 Bajo:</b> Score < 50</li>
        </ul>
    </div>
    '''),
    
    widgets.HTML('<h3 style="color: #E65100;">🚀 Generar Análisis</h3>'),
    widgets.HBox([generar_btn, analisis_excel_btn]),
    
    widgets.HTML('''
    <div style="background: #E3F2FD; padding: 12px; border-radius: 8px; margin: 10px 0; border-left: 4px solid #2196F3;">
        <h4 style="margin: 0 0 8px 0; color: #1565C0;">📊 Análisis Excel Incluye:</h4>
        <ul style="margin: 0; padding-left: 20px; font-size: 12px; color: #1976D2;">
            <li><b>🏆 Resumen METs:</b> Rankings y estadísticas consolidadas por MET</li>
            <li><b>🛣️ Resumen RUTAs:</b> Análisis detallado por ruta con rankings comparativos</li>
            <li><b>📋 Detalle Clientes:</b> Información completa de todos los clientes con rankings</li>
            <li><b>⚡ Análisis de Eficiencia:</b> Ratios de rentabilidad por MET y ruta</li>
            <li><b>📊 Métricas Comparativas:</b> Rankings múltiples para identificar mejores performers</li>
        </ul>
    </div>
    '''),
    
    output_map,
    output_excel], layout=widgets.Layout(padding='20px')))